# Advanced Time Series Forecasting, Optimization and Explainability

## Final Project - Advanced Topics in Deep Learning (2025/2026)

## 1. Introduction and Project Goals

This project builds upon the Time Series Modelling mini-project to develop a systematic, optimized, and well-analysed forecasting system for air temperature prediction using the Jena Climate dataset. While the mini-project focused on exploratory modelling and understanding different design choices, this Final Project elevates performance to a primary objective, achieved through systematic optimization and supported by rigorous analysis.

The core forecasting task is a **multivariate-input, univariate-output, multi-step prediction** problem: given a window of historical meteorological measurements (temperature, pressure, humidity, wind speed, maximum wind speed, and wind direction), the system predicts the air temperature for the next 24 hours.

Our pipeline integrates the following advanced techniques:

- **Baseline Deep Learning Model (GRU):** A configurable multi-layer GRU architecture serves as the reference model against which all improvements are measured, incorporating regularization, gradient clipping, and modern training strategies (AdamW optimizer, Huber loss).

- **Evolutionary Optimization:** A genetic algorithm (GA) automatically searches over an end-to-end pipeline configuration space — jointly optimizing model hyperparameters, architectural choices, training settings, preprocessing strategy, regularization, and windowing parameters. With a population of 20 individuals over 15 generations (300 total evaluations), the EA discovers configurations that outperform the hand-tuned baseline while using fewer parameters.

- **Synthetic Data Generation (TimeGAN):** A Time-series Generative Adversarial Network generates realistic synthetic weather sequences from the training set, demonstrating the feasibility of data augmentation without information leakage.

- **Explainable AI (XAI):** Both global (permutation importance) and local (gradient saliency) explanation methods are applied. XAI insights are further used to guide **feature pruning**, removing redundant features to produce a simpler model with improved performance and robustness.

- **Efficiency and Resource Analysis:** Training time, inference latency, parameter counts, and memory usage (RAM and GPU) are profiled across model configurations, enabling an informed analysis of accuracy–efficiency trade-offs.

The remainder of this report is organized as follows: Section 2 describes the experimental setup and reproducibility measures; Section 3 presents the dataset and problem formulation; Sections 4–6 cover data preparation, feature engineering, and the windowing pipeline; Section 7 establishes the baseline models; Section 8 details the TimeGAN synthetic data generation; Section 9 presents the evolutionary optimization process; Section 10 covers final model retraining, selection, and robustness validation; Sections 11–12 present the explainability and efficiency analyses; and Sections 13–14 provide the comparative discussion and conclusion.

## 2. Environment, Reproducibility and Experimental Setup

*Methodology — This section and Sections 3–6 describe the data preparation and experimental methodology used throughout the project.*

This section establishes the computational environment, library dependencies, random seed configuration, and project structure. Reproducibility is ensured through fixed seeds, explicit dependency management, and modular code organization.

### 2.1 Libraries Import

In [ ]:
import os
import gc
import sys
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import optuna
import psutil

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

### GPU Test

We verify that TensorFlow detects the available GPU and configure memory growth to prevent out-of-memory errors during the evolutionary search, which trains many models sequentially.

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

### 2.2 Utils Configuration Test

We set a global random seed (`SEED = 42`) applied consistently to Python, NumPy, and TensorFlow via `set_global_seed()` to ensure reproducibility. GPU memory growth is enabled via `enable_gpu_memory_growth()`, and device information is printed to confirm the hardware configuration.

In [ ]:
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

Set up the project root path and import custom utility functions.

In [ ]:
from src.utils.env import set_global_seed, enable_gpu_memory_growth, get_device_info

SEED = 42
set_global_seed(SEED)
enable_gpu_memory_growth()

print(get_device_info())

We import our custom utility functions and verify the environment setup (seed, GPU memory growth, device info).

## 3. Dataset and Problem Definition

The Jena Climate dataset provides the empirical basis for all experiments conducted in this project. This section presents the dataset, the initial data quality checks, the temporal resampling procedure to hourly resolution, and the formal definition of the forecasting problem, including the target variable and the selected input variables.

### 3.1 Load Data

The Jena Climate dataset is loaded from a local CSV file. The dataset contains meteorological measurements collected at a weather station in Jena, Germany, between January 2009 and December 2016, with an original sampling frequency of 10 minutes across 15 variables.

In [ ]:
from src.data.ingestion import ingest

PROJECT_ROOT = Path.cwd().parent
DATASET_PATH = str(PROJECT_ROOT / "data" / "jena_climate_2009_2016.csv")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_PATH:", DATASET_PATH)

Load the dataset CSV and preview the first rows.

**Table 1** — First rows of the raw Jena Climate dataset (10-minute sampling, 15 variables).

In [ ]:
df, ingest_summary = ingest(DATASET_PATH)

print("Ingestion summary:")
for k, v in ingest_summary.items():
    print(f"  {k}: {v}")

df.head()

Load the CSV file and preview the first rows to confirm the data was read correctly.

### 3.2 Initial Dataset Inspection

Upon loading, we inspect the dataset shape, column names, data types, and general structure. This provides a first look at the raw data before any transformations are applied.

In [ ]:
print("Dataset path:", DATASET_PATH)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nInfo:")
df.info()

After loading the raw dataset, we perform quality checks to ensure data integrity before any transformations. Specifically, we verify the presence of duplicated rows and inspect all columns for missing values. Any duplicated observations are removed to avoid biasing the resampling step that follows, duplicate timestamps can arise from logging artifacts and must be eliminated to ensure a consistent temporal index.

In [ ]:
print("Duplicated rows:", df.duplicated().sum())
print("\nMissing values per column:")
print(df.isna().sum())

In [ ]:
# Deduplication already handled by ingest()
print("Shape after ingestion:", df.shape)
print("Duplicated rows:", df.duplicated().sum())

Remove duplicated rows and verify the cleanup.

### 3.3 Datetime Parsing and Temporal Ordering

The `Date Time` column is parsed into proper datetime objects and the dataframe is sorted chronologically. We verify the date range and inspect the time delta distribution to detect any irregularities in the 10-minute sampling frequency before resampling.

**Table 2** — Dataset date range and first rows after datetime parsing and temporal ordering.

In [ ]:
# Datetime parsing already handled by ingest()
print(df["Date Time"].min(), "->", df["Date Time"].max())
df[["Date Time"]].head()

Check the time delta distribution to identify sampling irregularities.

In [ ]:
df["Date Time"].diff().value_counts().head(10)

Inspect the distribution of time deltas between consecutive observations to detect any sampling irregularities.

### 3.4 Hourly Resampling

After removing duplicated rows, the cleaned dataset contained 420,224 observations at approximately 10-minute resolution. To align the forecasting problem with hourly dynamics and reduce computational cost, the series was resampled to **1-hour intervals** using `resample("1h").mean()`, which aggregates all observations within each hourly bin through arithmetic averaging.

This operation reduced the dataset from **420,224** rows to **70,129** rows, approximately a sixfold reduction, while preserving the dominant meteorological patterns relevant for day-ahead forecasting. Mean aggregation was preferred over alternatives such as first/last value selection or median aggregation because it preserves the central tendency of each hourly period while smoothing sub-hourly noise that is less relevant for the 24-hour prediction horizon.

In [ ]:
from src.data.preprocessing import resample_hourly, select_features, temporal_split

df_hourly, n_nan_dropped = resample_hourly(df)

print("Original shape:", df.shape)
print("Hourly shape:", df_hourly.shape)
print("NaN rows dropped:", n_nan_dropped)
print(df_hourly.head())

### 3.5 Post-Resampling Quality Check

After hourly resampling, the dataset was re-validated to ensure that the transformation did not introduce structural artifacts. In particular, we verified that no duplicated hourly timestamps were created, that the time difference between consecutive observations was predominantly **1 hour**, and that any missing values produced by incomplete hourly bins were explicitly identified.

The quality check showed that the resampled dataset contained **88 missing values per meteorological variable**, while the `Date Time` column remained complete. No duplicated timestamps were found after resampling, and the time-step distribution confirmed that the series was overwhelmingly regular at **1-hour intervals**. This validation step is essential because the supervised sliding-window procedure used later assumes a temporally ordered and nearly regular time series.

In [ ]:
print("Missing values after hourly resampling:")
print(df_hourly.isna().sum())

print("\nDuplicated timestamps after resampling:", df_hourly["Date Time"].duplicated().sum())

print("\nTime step distribution after resampling:")
print(df_hourly["Date Time"].diff().value_counts().head(10))

### 3.6 Handling Missing Values After Resampling

All rows containing NaN values after hourly resampling were removed. Because the original dataset has near-complete 10-minute coverage across the 2009–2016 period, the number of affected rows was very small relative to the full hourly dataset. After dropping these rows, the dataset size became **70,041 observations**, and all remaining missing values were eliminated.

Dropping these observations was preferred over imputation because the affected hourly bins were sparse and associated with incomplete aggregation windows or isolated temporal irregularities. In such cases, interpolation could introduce artificial patterns into the series and potentially bias the forecasting models. Although the cleaned series still contains a very small number of larger temporal gaps, the dataset remains overwhelmingly regular at hourly frequency and is suitable for the supervised windowing procedure used in the next stages.

In [ ]:
# NaN handling already done by resample_hourly()
print("Shape after resampling:", df_hourly.shape)
print("Missing values:", df_hourly.isna().sum().sum())

### 3.7 Forecasting Task and Variable Selection

Following the project specification, we retain 6 meteorological variables plus the datetime reference: `T (degC)` (air temperature, the target), `p (mbar)` (atmospheric pressure), `rh (%)` (relative humidity), `wv (m/s)` (wind speed), `max. wv (m/s)` (maximum wind speed), and `wd (deg)` (wind direction). All other variables from the original 15-column dataset are excluded.

This defines a **multivariate-input, univariate-output, multi-step** forecasting problem: the model receives a multi-day window of multi-sensor weather data and must predict the air temperature trajectory for the next 24 hours. The multivariate input ensures the model can leverage cross-variable dependencies — for instance, the relationship between falling pressure and subsequent temperature changes driven by weather fronts.

**Table 3** — Selected variables for the forecasting task (6 meteorological features + datetime).

In [ ]:
# Feature selection from Hydra config (cfg injected by noted)
selected_columns = list(cfg.data.features)
if "Date Time" not in selected_columns:
    selected_columns = ["Date Time"] + selected_columns

df_model = select_features(df_hourly, selected_columns)

print("Selected columns:")
print(df_model.columns.tolist())
print("\nShape:", df_model.shape)
df_model.head()

## 4. Data Initial Preparation

Before applying feature engineering, the core modelling dataframe is established by defining the target variable and confirming the structure and integrity of the cleaned dataset. This section bridges the data cleaning stage presented in Section 3 with the feature engineering pipeline developed in Section 5, ensuring a clear and consistent transition from raw observations to model-ready inputs.

### 4.1 Target Definition and Base Modeling DataFrame

The target variable is defined as `T (degC)` — air temperature in degrees Celsius. This is the only quantity the model must forecast in the output window; all other variables serve as exogenous inputs.

The `Date Time` column serves as the temporal reference for feature engineering (extracting hour-of-day and day-of-year), but is not included as a direct model input since neural networks cannot interpret raw datetime objects. Instead, its temporal information is encoded as cyclic features in Section 5.

The remaining 5 meteorological variables — atmospheric pressure, relative humidity, wind speed, maximum wind speed, and wind direction — form the covariate set that provides the atmospheric context for the temperature forecast.

In [ ]:
TARGET_COL = cfg.data.target
TIME_COL = "Date Time"

feature_cols = [col for col in df_model.columns if col not in [TIME_COL]]
input_feature_cols = [col for col in feature_cols]

print("Target:", TARGET_COL)
print("Time column:", TIME_COL)
print("Input features:", input_feature_cols)
print("Number of input features:", len(input_feature_cols))

### 4.2 Base Forecasting Data Overview

We inspect the prepared dataframe to confirm shape, date range, and column integrity before proceeding to feature engineering. The dataset at this stage contains approximately 70,000 hourly observations spanning from January 2009 to January 2017, with 7 columns (datetime + 6 meteorological variables). This verification step ensures that no data was inadvertently lost during the resampling and cleaning pipeline of Section 3, and establishes the baseline from which all derived features will be computed.

In [ ]:
print(df_model.head())
print("\nShape:", df_model.shape)
print("\nDate range:", df_model[TIME_COL].min(), "->", df_model[TIME_COL].max())

## 5. Feature Engineering

Two families of derived features are introduced to enrich the input representation: **cyclical temporal encodings** and **wind-derived features**. The goal is to provide the neural network with representations that respect the physical structure of the data, cyclic quantities are encoded cyclically, and vector quantities are decomposed into Cartesian components. All feature engineering is implemented in `src.features.engineering` for reproducibility.

In [ ]:
from src.features.engineering import (
    add_time_features,
    add_wind_features,
    get_final_feature_columns,
)
from src.features.windowing import make_windows
from src.models.gru import build_gru_model

### 5.1 Cyclical Time Features

Hour-of-day and day-of-year carry strong periodic signals (diurnal and seasonal cycles). Encoding them as raw integers would introduce artificial discontinuities (e.g., hour 23 far from hour 0). We apply a sine–cosine encoding that maps each cyclic variable to a point on the unit circle, ensuring smooth transitions at boundaries:
- `hour_sin`, `hour_cos` — diurnal cycle (period = 24h)
- `doy_sin`, `doy_cos` — seasonal cycle (period = 365.25 days)

**Table 4** — Cyclical temporal features derived from hour-of-day and day-of-year.

In [ ]:
df_feat = add_time_features(df_model, time_col=TIME_COL)

df_feat[[
    TIME_COL, "hour", "dayofyear",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos"
]].head()

### 5.2 Wind-Derived Features

Wind direction in degrees is inherently circular and difficult for neural networks to interpret directly. We derive six features:
- `wd_sin`, `wd_cos` — cyclic encoding of wind direction
- `wx`, `wy` — Cartesian decomposition of wind vector (speed × cos/sin of direction), replacing the polar representation with a form neural networks process more naturally
- `wind_gap` — difference between max and sustained wind speed (gust intensity)
- `gust_ratio` — ratio of max to sustained wind speed (relative gust strength, with ε = 10⁻⁶ for numerical stability)

**Table 5** — Wind-derived features: cyclic encoding, Cartesian components, gust metrics.

In [ ]:
df_feat = add_wind_features(df_feat)

df_feat[[
    TIME_COL, "wv (m/s)", "max. wv (m/s)", "wd (deg)",
    "wd_sin", "wd_cos", "wx", "wy", "wind_gap", "gust_ratio"
]].head()

### 5.3 Final Feature Set for Modeling

The complete input feature vector contains **16 dimensions**: 6 original meteorological variables + 4 cyclical temporal encodings + 6 wind-derived features. This feature set is used consistently across all experiments, baseline, evolutionary optimization, and synthetic data generation. The XAI analysis in Section 11 later evaluates which of these 16 features contribute most, leading to a pruned 11-feature variant.

In [ ]:
final_feature_cols = get_final_feature_columns()

print("Number of modeling features:", len(final_feature_cols))
print(final_feature_cols)

## 6. Split, Scaling and Windowing

This section completes the data preparation pipeline by defining the temporal train/validation/test split, applying feature scaling without data leakage, and transforming the time series into supervised learning windows. These steps produce the final input–output tensors used by all forecasting models throughout the project.

### 6.1 Temporal Train/Validation/Test Split

The dataset is split chronologically (70% / 15% / 15%) with **no shuffling**, the split respects temporal order to prevent data leakage. The training set covers the earliest years, followed by validation, with the most recent data reserved for testing. This mirrors a realistic deployment scenario where the model is trained on historical data and evaluated on future observations.

In [ ]:
df_train, df_val, df_test = temporal_split(
    df_feat,
    train_ratio=cfg.data.split.train,
    val_ratio=cfg.data.split.val,
)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

### 6.2 Feature Scaling

All input features are normalized using **StandardScaler** (zero mean, unit variance) as the baseline strategy. The scaler is fit exclusively on the training set and applied via `transform` to validation and test sets, preventing information leakage. The evolutionary optimization also searches over the scaler type (standard, robust, minmax), allowing the GA to discover whether a different normalization strategy benefits specific model configurations.

In [ ]:
from src.features.scaling import get_scaler

Apply the selected scaler to all splits (fit on train, transform on val/test).

In [ ]:
SCALER_NAME = cfg.scaler.name
scaler = get_scaler(SCALER_NAME)

X_train_df = df_train[final_feature_cols].copy()
X_val_df = df_val[final_feature_cols].copy()
X_test_df = df_test[final_feature_cols].copy()

X_train_scaled = scaler.fit_transform(X_train_df)
X_val_scaled = scaler.transform(X_val_df)
X_test_scaled = scaler.transform(X_test_df)

print("Scaler:", SCALER_NAME)
print("Scaled train shape:", X_train_scaled.shape)
print("Scaled val shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Apply the scaler to the training, validation, and test feature matrices. The scaler is fit only on the training data.

### 6.3 Target Index for Forecasting

We identify the position of `T (degC)` within the 16-feature vector. This index is used during windowing to extract only the temperature column for the output sequence **y**, while keeping all 16 features in the input tensor **X**. This separation makes the problem multivariate-input but univariate-output.

In [ ]:
target_idx = final_feature_cols.index(TARGET_COL)

print("Target column:", TARGET_COL)
print("Target index:", target_idx)

### 6.4 Supervised Windowing

The scaled multivariate time series is converted into supervised learning samples through a sliding-window procedure. For each sample, the input tensor **X** contains a sequence of consecutive hourly observations across all modeling features, while the output vector **y** contains only the future air temperature values to be predicted.

Under the default configuration, the input has shape **`(LOOKBACK, 16)`** and the output has shape **`(HORIZON,)`**, corresponding to a multivariate-input, univariate-output, multi-step forecasting setup. In the base configuration, `LOOKBACK = 120` hours (5 days) and `HORIZON = 24` hours (1 day ahead). In the evolutionary optimization stage presented in Section 9, alternative lookback lengths are also explored as part of the search space, so the windowing strategy is not treated as fixed throughout the project.

In [ ]:
from src.features.windowing import make_windows

Create supervised windows for all splits using the defined lookback and horizon.

In [ ]:
LOOKBACK = cfg.data.lookback
HORIZON = cfg.data.horizon

X_train, y_train = make_windows(X_train_scaled, target_idx, LOOKBACK, HORIZON)
X_val, y_val = make_windows(X_val_scaled, target_idx, LOOKBACK, HORIZON)
X_test, y_test = make_windows(X_test_scaled, target_idx, LOOKBACK, HORIZON)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

Create supervised windows for train, validation, and test splits using the defined lookback (120h) and horizon (24h).

### 6.5 Windowed Data Sanity Check

A final sanity check is performed on the windowed data to confirm that the supervised transformation produced tensors with the expected structure and dimensionality. In particular, the input windows follow the shape **`(N, LOOKBACK, 16)`**, while the target arrays follow the shape **`(N, HORIZON)`**, corresponding to multivariate input sequences and 24-step univariate temperature targets.

The inspection confirms that each input sample contains **120 hourly observations across 16 features**, and that each target sample contains **24 future temperature values**. This validation step is important because it helps detect off-by-one errors or alignment mistakes in the windowing process that could silently compromise model training and evaluation.

In [ ]:
print("Input shape:", X_train.shape[1:])
print("Forecast horizon:", y_train.shape[1])
print("Number of input features:", X_train.shape[2])

print("\nExample input window shape:", X_train[0].shape)
print("Example target shape:", y_train[0].shape)
print("First 5 target values (scaled):", y_train[0][:5])

## 7. Baseline Models

Sections 7 to 9 present the core experimental pipeline of the project, covering baseline establishment, synthetic data generation, and evolutionary optimization.

Two baseline models are defined as reference points for all subsequent analyses. The **persistence baseline** provides a minimal no-skill benchmark that any learned forecasting model should outperform. The **GRU baseline** represents a stronger deep learning reference model, trained under a modern and carefully controlled setup, and serves as the main benchmark against which the optimized evolutionary solutions are evaluated.

### 7.1 Persistence Baseline

The persistence baseline corresponds to the simplest possible forecasting strategy: the last observed temperature value is repeated across the entire 24-hour forecast horizon. This naive approach provides a minimal reference level for performance, since any learned forecasting model should be able to outperform it.

The resulting prediction tensor has shape **`(10364, 24)`**, matching the multi-step forecasting target, and each prediction consists of a constant repetition of the final observed temperature from the corresponding input window.

In [ ]:
def persistence_forecast(X):
    # Repeats the last observed target value across the full forecast horizon
    last_temp = X[:, -1, target_idx]
    return np.repeat(last_temp[:, None], HORIZON, axis=1)

y_pred_persistence = persistence_forecast(X_test)

print("Persistence prediction shape:", y_pred_persistence.shape)
print("First prediction:", y_pred_persistence[0][:5])

Compute scaled metrics for the persistence forecast on the test set.

In [ ]:
persistence_mae_scaled = mean_absolute_error(y_test.flatten(), y_pred_persistence.flatten())
persistence_rmse_scaled = np.sqrt(mean_squared_error(y_test.flatten(), y_pred_persistence.flatten()))

print("Persistence MAE (scaled):", persistence_mae_scaled)
print("Persistence RMSE (scaled):", persistence_rmse_scaled)

Compute persistence baseline metrics on the test set.

In [ ]:
def build_gru_baseline(input_shape, horizon):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dense(horizon)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

gru_baseline = build_gru_baseline(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    horizon=HORIZON
)

gru_baseline.summary()

Before building the official configurable baseline, we construct a simple 1-layer GRU (64 units) as a quick sanity check. This verifies that the full pipeline — from windowed data through model training to test evaluation — works correctly end-to-end. The simple model's results are not used for comparison; they serve only as a development checkpoint.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_baseline = gru_baseline.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Train the simple GRU with early stopping on validation loss.

In [ ]:
y_pred_gru = gru_baseline.predict(X_test, verbose=0)

print("Prediction shape:", y_pred_gru.shape)
print("First prediction:", y_pred_gru[0][:5])

Generate and inspect test set predictions from the simple GRU.

In [ ]:
gru_mae_scaled = mean_absolute_error(y_test.flatten(), y_pred_gru.flatten())
gru_rmse_scaled = np.sqrt(mean_squared_error(y_test.flatten(), y_pred_gru.flatten()))

print("GRU MAE (scaled):", gru_mae_scaled)
print("GRU RMSE (scaled):", gru_rmse_scaled)

Evaluate the simple GRU on scaled metrics to compare against persistence.

### 7.2 GRU Baseline

The official GRU baseline was defined using the best-performing configuration identified in the previous Time Series Modelling mini-project. This ensures methodological continuity and provides a strong, previously validated reference model for the current project.

The model is implemented through `build_gru_model()` from `src.models.gru` and consists of a configurable 2-layer GRU architecture with **96** and **64** hidden units, respectively. The baseline also includes an intermediate dense layer with **256** units, **AdamW** optimization, **Huber loss** with \( \delta = 1.0 \), **L2 regularization**, and **gradient clipping**. Training is performed with **early stopping** (`patience=6`) and **learning rate reduction on plateau** (`patience=3`, `factor=0.5`), providing a stable and competitive deep learning benchmark against which the evolutionary optimization will be evaluated.

In [ ]:
from src.models.gru import build_gru_model

Define the official baseline configuration and build the model.

In [ ]:
from src.training.pipeline import build_model_from_cfg, train_pipeline

# Load model config from Hydra (cfg.model contains the active model config)
BASELINE_CFG = {
    "n_layers": cfg.model.n_layers,
    "units1": cfg.model.units1,
    "units2": cfg.model.units2,
    "units3": cfg.model.units3,
    "dropout": cfg.model.dropout,
    "l2": cfg.model.l2,
    "dense_units": cfg.model.dense_units,
    "dense_activation": cfg.model.dense_activation,
    "learning_rate": cfg.training.learning_rate,
    "clipnorm": cfg.training.clipnorm,
    "optimizer_name": cfg.model.optimizer_name,
    "weight_decay": cfg.model.weight_decay,
    "loss_name": cfg.model.loss_name,
    "gaussian_noise_std": cfg.model.gaussian_noise_std,
    "batch_size": cfg.training.batch_size,
    "scaler_name": cfg.scaler.name,
}

gru_baseline = build_model_from_cfg(BASELINE_CFG, LOOKBACK, X_train.shape[2], HORIZON)
gru_baseline.summary()

In [ ]:
from src.models.train_eval import (
    train_model,
    evaluate_scaled_forecasts,
    inverse_scale_target,
    evaluate_original_scale_forecasts,
)

Import training and evaluation utilities.

In [ ]:
history_baseline = train_model(
    model=gru_baseline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    batch_size=cfg.training.batch_size,
    epochs=cfg.training.epochs,
    verbose=1
)

The official baseline is trained with the full 60-epoch budget. Training uses early stopping (patience=6, restoring best weights) and learning rate reduction on plateau (patience=3, factor=0.5), ensuring the model converges without overfitting.

In [ ]:
y_pred_gru = gru_baseline.predict(X_test, verbose=0)

print("Prediction shape:", y_pred_gru.shape)
print("First prediction:", y_pred_gru[0][:12])

Generate test predictions from the official baseline and inspect the output.

In [ ]:
gru_scaled_metrics = evaluate_scaled_forecasts(y_test, y_pred_gru)

gru_mae_scaled = gru_scaled_metrics["mae_scaled"]
gru_rmse_scaled = gru_scaled_metrics["rmse_scaled"]

print("GRU Baseline MAE (scaled):", gru_mae_scaled)
print("GRU Baseline RMSE (scaled):", gru_rmse_scaled)

Compute scaled MAE and RMSE for the official GRU baseline on the test set.

### 7.3 Baseline Comparison

Table 6 presents a side-by-side comparison of the persistence baseline and the official GRU baseline using **scaled evaluation metrics**, namely MAE and RMSE computed in normalized space. These metrics are useful for comparing models trained under the same scaling regime and for monitoring optimization behaviour during training.

However, scaled errors do not have a direct physical interpretation. For this reason, the comparison is extended in the following subsections by converting both predictions and targets back to the original temperature scale, allowing the results to be interpreted in degrees Celsius.

**Table 6** — Persistence vs. GRU baseline comparison on scaled metrics (test set).

In [ ]:
baseline_results_scaled = pd.DataFrame({
    "Model": ["Persistence", "GRU Baseline Official"],
    "MAE_scaled": [persistence_mae_scaled, gru_mae_scaled],
    "RMSE_scaled": [persistence_rmse_scaled, gru_rmse_scaled],
})

baseline_results_scaled

### 7.4 Reverse Scaling

Predictions and ground truth are inverse-transformed from normalized space back to degrees Celsius using the training set's scaler statistics (mean and standard deviation of the temperature column). This allows evaluation in physically meaningful units.

In [ ]:
target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]

y_test_inv = inverse_scale_target(y_test, target_mean, target_std)
y_pred_persistence_inv = inverse_scale_target(y_pred_persistence, target_mean, target_std)
y_pred_gru_inv = inverse_scale_target(y_pred_gru, target_mean, target_std)

print("y_test_inv shape:", y_test_inv.shape)
print("y_pred_gru_inv shape:", y_pred_gru_inv.shape)

### 7.5 Baseline Evaluation in Original Temperature Scale

We compute MAE and RMSE in degrees Celsius on the inverse-scaled predictions. These original-scale metrics are the primary comparison basis throughout the report — an MAE of 1.65°C means the model's 24-hour forecasts are off by an average of 1.65 degrees. This is the metric used as the evolutionary fitness function and the final evaluation criterion.

In [ ]:
gru_original_metrics = evaluate_original_scale_forecasts(y_test_inv, y_pred_gru_inv)
persistence_original_metrics = evaluate_original_scale_forecasts(y_test_inv, y_pred_persistence_inv)

gru_mae = gru_original_metrics["mae"]
gru_rmse = gru_original_metrics["rmse"]

persistence_mae = persistence_original_metrics["mae"]
persistence_rmse = persistence_original_metrics["rmse"]

print("GRU Baseline MAE (°C):", gru_mae)
print("GRU Baseline RMSE (°C):", gru_rmse)

### 7.6 Final Baseline Comparison

Table 7, combines both **scaled** and **original-scale** evaluation metrics for the persistence and GRU baseline models. The official GRU baseline substantially outperforms the naive persistence forecast, reducing the test error from **3.144 °C** to **1.665 °C** in MAE and from **4.254 °C** to **2.209 °C** in RMSE.

These results confirm that the recurrent architecture is able to capture meaningful temporal dependencies in the meteorological series and establish a strong benchmark against which the subsequent optimization stages will be evaluated.

**Table 7** — Final baseline comparison with both scaled and original-scale (°C) metrics.

In [ ]:
baseline_results = pd.DataFrame({
    "Model": ["Persistence", "GRU Baseline Official"],
    "MAE_scaled": [persistence_mae_scaled, gru_mae_scaled],
    "RMSE_scaled": [persistence_rmse_scaled, gru_rmse_scaled],
    "MAE_degC": [persistence_mae, gru_mae],
    "RMSE_degC": [persistence_rmse, gru_rmse],
})

baseline_results

### 7.7 Forecast Visualization

Figure 1, presents a visual comparison of a single 24-hour forecast window on the test set, showing the true future temperature trajectory together with the persistence and GRU baseline predictions in degrees Celsius.

The persistence baseline produces a flat forecast, since it simply repeats the last observed temperature across the full horizon. As expected, this strategy fails to capture the strong upward and downward variations visible in the true trajectory. In contrast, the GRU baseline is able to follow the general temporal pattern more closely, correctly anticipating the broad rise in temperature during the first half of the horizon and the subsequent declining trend.

Although the GRU forecast is smoother than the true signal and underestimates the amplitude of some changes, it still represents a substantial improvement over persistence. This qualitative comparison is consistent with the quantitative results reported earlier and confirms that the recurrent model captures meaningful temporal structure in the meteorological series.

In [ ]:
sample_idx = 0

plt.figure(figsize=(10, 5))
plt.plot(y_test_inv[sample_idx], label="True", marker="o")
plt.plot(y_pred_persistence_inv[sample_idx], label="Persistence", linestyle="--")
plt.plot(y_pred_gru_inv[sample_idx], label="GRU Baseline", linestyle="--")
plt.title("Example Multi-step Forecast on Test Set")
plt.xlabel("Forecast Step")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)
plt.show()

**Figure 1** — 24-hour forecast comparison: ground truth vs. persistence vs. GRU baseline (°C).

## 8. Synthetic Data Generation with TimeGAN

This section explores generative approaches for creating realistic synthetic weather sequences, with the goal of demonstrating data augmentation feasibility without information leakage.

### 8.1 Motivation and Experimental Role

The purpose of the TimeGAN component is to generate realistic synthetic multivariate weather sequences using only the training split, thereby avoiding any form of data leakage.

These synthetic sequences are intended for training-set augmentation, allowing us to assess whether synthetic data can improve forecasting performance, robustness, or generalization.

To remain consistent with the forecasting setup, the generative model is trained on multivariate sequences of length **`LOOKBACK + HORIZON`**, so that each generated sequence can later be divided into:
- an input window of length **`LOOKBACK`**
- a target forecasting horizon of length **`HORIZON`**

The impact of synthetic augmentation is then evaluated by comparing forecasting models trained under two conditions:
1. using real data only
2. using real data together with synthetic data

### 8.2 TimeGAN Training Data Preparation

Training data is converted into fixed-length sequences of `LOOKBACK + HORIZON = 144` time steps, matching the forecasting window. This ensures generated sequences can be directly split into input–output pairs. Only the **training set** is used — validation and test data are never exposed to the generative model, preventing data leakage.

In [ ]:
from src.gan.data_prep import make_timegan_sequences, split_synthetic_sequences

TIMEGAN_SEQ_LEN = LOOKBACK + HORIZON

timegan_train_sequences = make_timegan_sequences(X_train_scaled, TIMEGAN_SEQ_LEN)

print("TimeGAN sequence length:", TIMEGAN_SEQ_LEN)
print("TimeGAN training sequences shape:", timegan_train_sequences.shape)

### 8.3 TimeGAN Model and Training Procedure

TimeGAN (Yoon et al., 2019) consists of five GRU-based sub-networks: **Embedder** (data → latent), **Recovery** (latent → data), **Generator** (noise → latent), **Supervisor** (temporal dynamics in latent space), and **Discriminator** (real vs. synthetic). Each uses 3 GRU layers with hidden dimension 24. Training follows a three-phase curriculum: autoencoder pretraining, supervisor pretraining, and adversarial training.

In [ ]:
from src.gan.config import TIMEGAN_CONFIG
from src.gan.timegan import TimeGAN

TIMEGAN_CONFIG["seq_len"] = TIMEGAN_SEQ_LEN

timegan = TimeGAN(
    seq_len=TIMEGAN_CONFIG["seq_len"],
    n_features=timegan_train_sequences.shape[2],
    hidden_dim=TIMEGAN_CONFIG["hidden_dim"],
    num_layers=TIMEGAN_CONFIG["num_layers"],
    learning_rate=TIMEGAN_CONFIG["learning_rate"],
    gamma=TIMEGAN_CONFIG["gamma"],
)

print(TIMEGAN_CONFIG)
print("TimeGAN input shape:", timegan_train_sequences.shape)

Print the full architecture summary of the TimeGAN model components.

In [ ]:
timegan.summary()

Print the architecture summary of all five TimeGAN sub-networks.

#### 8.3.1 Autoencoder Pretraining

**Phase 1:** The embedder and recovery networks are jointly trained to minimize reconstruction error (MSE) on real sequences. This stage is intended to learn a meaningful latent representation of the multivariate weather dynamics before adversarial training begins.

The reconstruction loss decreases steadily from approximately **0.280** in the first epoch to **0.0023** by epoch 20, indicating that the autoencoder is able to compress and reconstruct the training sequences with high fidelity, see figure 2. This suggests that the latent space learned by the embedder is sufficiently informative to support the next stages of TimeGAN training.

In [ ]:
autoencoder_history = timegan.pretrain_autoencoder(
    sequences=timegan_train_sequences,
    epochs=TIMEGAN_CONFIG["ae_epochs"],
    batch_size=TIMEGAN_CONFIG["batch_size"],
    verbose=1,
)

Visualize the autoencoder pretraining loss curve.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(autoencoder_history.history["loss"], label="Autoencoder Train Loss")
plt.title("TimeGAN Autoencoder Pretraining Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.legend()
plt.show()

**Figure 2** — TimeGAN autoencoder pretraining loss (MSE) over 20 epochs.

Visualize the autoencoder pretraining loss curve.

#### 8.3.2 Supervisor Pretraining

**Phase 2:** The supervisor network is trained to predict the next latent time step from the current one, using latent representations produced by the embedder. This stage is designed to capture temporal dynamics in latent space before full adversarial optimization begins.

The supervisor loss decreases from approximately **0.0308** in the first epoch to **0.0076** by epoch 20, with the largest improvement occurring during the initial epochs and a gradual stabilization thereafter, see figure 3. This behaviour suggests that the latent temporal dynamics are being learned successfully, although convergence is less pronounced than in the autoencoder stage. Such a pattern is expected, since temporal prediction in latent space is typically more challenging than direct reconstruction.

In [ ]:
supervisor_history = timegan.pretrain_supervisor(
    sequences=timegan_train_sequences,
    epochs=TIMEGAN_CONFIG["sup_epochs"],
    batch_size=TIMEGAN_CONFIG["batch_size"],
    verbose=1,
)

#### 8.3.3 Pretraining Loss Curves

Combined visualization of autoencoder and supervisor pretraining losses to verify both networks converged before adversarial training.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(autoencoder_history.history["loss"], label="Autoencoder Loss")
plt.plot(supervisor_history.history["loss"], label="Supervisor Loss")
plt.title("TimeGAN Pretraining Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.legend()
plt.show()

**Figure 3** — Combined pretraining loss curves for autoencoder and supervisor networks.

#### 8.3.4 Adversarial Training

**Phase 3:** The generator and discriminator are optimized adversarially. The total generator objective combines three components: an adversarial loss that encourages the generator to fool the discriminator, a supervised loss that promotes temporal coherence in latent space (weighted by **×100**), and a reconstruction loss that encourages fidelity in data space (weighted by **γ = 1.0**). A discriminator update threshold is used to reduce the risk of the discriminator becoming dominant too early in training.

During adversarial optimization, the discriminator loss remains within a moderate range, while the generator loss shows noticeable fluctuations across epochs. The supervised latent-dynamics component remains relatively stable and low, whereas the adversarial and reconstruction-related terms vary more substantially. This behaviour suggests that training remained numerically stable, but that the adversarial game did not converge as smoothly as the earlier pretraining stages.

The adversarial phase indicates that the model learned some generative structure without collapsing, but the loss trajectories also suggest that synthetic quality should be assessed carefully through downstream evaluation rather than inferred from adversarial losses alone.

In [ ]:
adversarial_history = timegan.fit(
    sequences=timegan_train_sequences,
    epochs=TIMEGAN_CONFIG["adv_epochs"],
    batch_size=TIMEGAN_CONFIG["batch_size"],
    verbose=1,
)

#### 8.3.5 Adversarial Training Loss Curves

Figure 4, visualizes the evolution of discriminator and generator losses during the adversarial training phase. In principle, a well-behaved TimeGAN should exhibit a dynamic balance between generator and discriminator, without one network collapsing or overwhelmingly dominating the other.

In this experiment, the discriminator loss remains within a moderate range, while the generator total loss fluctuates more noticeably across epochs. The supervised component stays consistently low, suggesting that latent temporal coherence is being preserved, whereas the adversarial and reconstruction-related terms remain more variable. This pattern indicates that training remained stable enough to continue, but did not reach a clearly smooth adversarial equilibrium.

Therefore, the loss curves suggest partial learning of the generative objective, but they also reinforce the need to evaluate synthetic sequence quality directly before drawing conclusions about the usefulness of TimeGAN-based augmentation.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(adversarial_history["d_loss"], label="Discriminator Loss")
plt.plot(adversarial_history["g_loss"], label="Generator Total Loss")
plt.plot(adversarial_history["g_loss_u"], label="Generator Adversarial Loss")
plt.plot(adversarial_history["g_loss_s"], label="Generator Supervised Loss")
plt.plot(adversarial_history["g_loss_v"], label="Generator Reconstruction Loss")
plt.title("TimeGAN Adversarial Training Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

**Figure 4** — TimeGAN adversarial training loss curves (discriminator, generator, supervised, reconstruction).

### 8.4 Synthetic Sequence Quality Assessment

Before considering synthetic data for augmentation, we assess whether the TimeGAN produces sequences that are statistically and temporally faithful to the real training data through reconstruction checks and visual comparisons.

#### 8.4.1 Autoencoder Reconstruction Check

We pass real sequences through the embedder → recovery path to verify the latent space captures meaningful structure. Low reconstruction MSE indicates the autoencoder learned a faithful representation.

In [ ]:
reconstructed_sequences = timegan.autoencoder.predict(
    timegan_train_sequences[:256],
    verbose=0
)

reconstruction_mse = np.mean((timegan_train_sequences[:256] - reconstructed_sequences) ** 2)

print("Reconstructed batch shape:", reconstructed_sequences.shape)
print("Reconstruction MSE on sample batch:", reconstruction_mse)

#### 8.4.2 Real vs Reconstructed Sequence

Figure 5, compares a real temperature sequence with its reconstruction produced by the TimeGAN autoencoder. The two trajectories show a close overall overlap across most of the 144 time steps, indicating that the learned latent representation retains the dominant temporal structure of the original series.

The reconstruction captures both the broad trend and most of the short-term fluctuations, although small deviations remain at some local peaks, turning points, and sharp declines. This behaviour is consistent with the low reconstruction loss observed during pretraining and suggests that the embedder–recovery pair is able to encode the input sequences with high fidelity. However, strong reconstruction quality alone does not guarantee high-quality synthetic generation, so the usefulness of TimeGAN must still be assessed through the generated samples and the downstream augmentation experiment.

In [ ]:
sample_idx = 0
feature_idx = 0  # T (degC)

plt.figure(figsize=(10, 4))
plt.plot(timegan_train_sequences[sample_idx, :, feature_idx], label="Real")
plt.plot(reconstructed_sequences[sample_idx, :, feature_idx], label="Reconstructed", linestyle="--")
plt.title("Real vs Reconstructed Sequence (Feature 0: T (degC))")
plt.xlabel("Time Step")
plt.ylabel("Scaled Value")
plt.grid(True)
plt.legend()
plt.show()

**Figure 5** — Real vs. reconstructed temperature sequence through the TimeGAN autoencoder.

#### 8.4.3 Synthetic Sequence Preview

A small batch of synthetic sequences is generated as an initial sanity check on the output of the trained generator. The generated batch has shape **`(8, 144, 16)`**, which matches the expected sequence length and feature dimensionality used throughout the forecasting pipeline.

The observed value range, from approximately **-1.003** to **0.776** in scaled space, indicates that the generator is producing bounded numerical outputs without obvious explosion or instability. However, this range is noticeably narrower than the variability typically observed in the real standardized data, suggesting that the synthetic sequences may still underrepresent part of the natural amplitude of the original series.

Therefore, this preview should be interpreted only as a basic sanity check. The actual usefulness of the generated sequences must be assessed through direct visual inspection and, more importantly, through the downstream comparison between forecasting models trained with and without synthetic augmentation.

In [ ]:
synthetic_sequences_preview = timegan.generate(8)

print("Synthetic preview shape:", synthetic_sequences_preview.shape)
print("Synthetic preview min:", synthetic_sequences_preview.min())
print("Synthetic preview max:", synthetic_sequences_preview.max())

#### 8.4.4 Real vs Synthetic Sequence

Figure 6, compares a real training sequence with a fully generated synthetic sequence for the temperature feature. This visualization provides a qualitative assessment of whether the generator is able to reproduce realistic temporal dynamics and value distributions after adversarial training.

Although the autoencoder reconstruction stage achieved high fidelity, the fully generated synthetic sequence does not adequately match the structure of the real signal. In particular, the synthetic trajectory shows much lower variability, reduced amplitude, and a flatter temporal pattern, failing to reproduce the richer dynamics observed in the real temperature series. This indicates that, despite successful latent reconstruction, the adversarial generation stage did not learn a sufficiently realistic data distribution.

The synthetic quality assessment suggests that the generated sequences are not yet reliable enough for data augmentation. While the autoencoder reconstruction is strong (with reconstruction MSE around **0.0012**), the synthetic samples themselves do not sufficiently replicate the temporal behaviour of the original data. For this reason, adding synthetic data to the forecasting pipeline would risk introducing noise and degrading predictive performance. Therefore, the TimeGAN component is retained as a proof-of-concept exploration of generative modelling, but synthetic augmentation was not used in the final forecasting models.

In [ ]:
sample_idx = 0
feature_idx = 0  # T (degC)

plt.figure(figsize=(10, 4))
plt.plot(timegan_train_sequences[sample_idx, :, feature_idx], label="Real sequence")
plt.plot(synthetic_sequences_preview[sample_idx, :, feature_idx], label="Synthetic sequence", linestyle="--")
plt.title("Real vs Synthetic Sequence After Adversarial Training")
plt.xlabel("Time Step")
plt.ylabel("Scaled Value")
plt.grid(True)
plt.legend()
plt.show()

**Figure 6** — Real training sequence vs. TimeGAN-generated synthetic sequence (temperature feature).

## 9. Evolutionary Optimization of the Forecasting Pipeline

### 9.1 Motivation and Search Strategy

The evolutionary optimization stage aims to improve the forecasting pipeline beyond the fixed GRU baseline by systematically exploring alternative architectural, preprocessing, training, and windowing configurations.

Instead of relying on a single manually selected model or on the Bayesian optimization procedure explored in previous work, this stage adopts a genetic search strategy based on:
- population initialization
- tournament selection
- crossover
- mutation
- elitism

Each candidate solution represents a full forecasting pipeline configuration, including GRU architecture, optimizer, loss function, regularization, batch size, scaling method, and input lookback length. The fitness of each candidate is evaluated on the **validation set only**, using forecasting performance in the original temperature scale, so that evolutionary selection remains aligned with the final project objective and avoids test-set leakage.

### 9.2 Search Space Definition

The evolutionary search space spans **17 optimization dimensions** grouped into five main categories:

- **Architecture:** number of GRU layers (`1–2`), hidden units per layer (`units1: 64–128`, `units2: 64–128`, `units3: 32–96`), optional dense layer (`0–256` units), and dense activation function (`ReLU`, `GELU`, `ELU`, `LeakyReLU`)
- **Regularization and optimization control:** dropout (`0.0–0.3`), L2 regularization (`0` to `10^{-4}`), Gaussian noise (`0.0` or `0.01`), and gradient clipping (`0.5–5.0`)
- **Training:** optimizer (`Adam`, `AdamW`), weight decay (`0` to `10^{-4}`), learning rate (`10^{-4}` to `10^{-3}`), loss function (`MSE`, `MAE`, `Huber δ=1`, `Huber δ=2`), and batch size (`128` or `256`)
- **Preprocessing:** scaler type (`standard`, `robust`, `minmax`)
- **Windowing:** input lookback length (`96`, `120`, `144` hours)

The final search space was deliberately narrowed relative to earlier exploratory versions in order to keep the evolutionary budget computationally tractable while still allowing meaningful variation in architecture, preprocessing, training dynamics, and temporal context length. In particular, the search was restricted to 1- and 2-layer GRU models and moderate hidden sizes, focusing computation on configurations more likely to be competitive for this dataset.

In [ ]:
from src.evolution.search_space import SEARCH_SPACE

print("Search space keys:")
print(list(SEARCH_SPACE.keys()))

for k, v in SEARCH_SPACE.items():
    print(f"{k}: {v}")

#### 9.2.1 Windowing Search Extension

Unlike the mini-project, the windowing configuration was not treated as fixed. In addition to architecture and training hyperparameters, the evolutionary search was extended to explore alternative lookback lengths while keeping the 24-hour forecast horizon fixed. This allowed the optimization process to jointly search over the forecasting model and the temporal context length used as input.

### 9.3 Evolutionary Representation and Constraints

Each evolutionary individual is represented as a **genotype**, implemented as a dictionary that maps gene names to admissible values from the predefined search space. This genotype encodes a full forecasting pipeline configuration, including architectural choices, optimization settings, preprocessing strategy, and input lookback length.

To ensure that sampled and evolved solutions remain valid and computationally meaningful, a set of constraints is enforced throughout the search process. In particular:
- GRU layer widths are constrained to be monotonically non-increasing (`units2 ≤ units1`, `units3 ≤ units2`), preventing unnecessarily inconsistent architectures
- `weight_decay` is forced to `0` when the selected optimizer is standard **Adam**, since explicit weight decay is only meaningful for **AdamW**
- all constraints are re-applied after initialization, crossover, and mutation, ensuring that every evaluated genotype corresponds to a valid forecasting configuration

This constrained representation helps reduce unproductive regions of the search space and improves the efficiency and fairness of the evolutionary search.

In [ ]:
from src.evolution.genotype import sample_genotype

example_genotype = sample_genotype()
example_genotype

### 9.4 Fitness Function

The fitness function encapsulates the full forecasting pipeline within a single evaluation procedure, from preprocessing and window construction to model training and validation forecasting. Each candidate genotype is decoded into a concrete forecasting configuration, trained on the training split, and evaluated exclusively on the validation split.

For each candidate configuration, the evaluation pipeline performs the following steps:
1. feature scaling using the candidate preprocessing strategy
2. supervised window generation using the candidate lookback length
3. GRU model construction according to the candidate architectural and training genes
4. model training on the training split
5. prediction on the validation split
6. computation of validation forecasting error

The evolutionary fitness is defined as the **validation MAE in the original temperature scale (°C)**. Scaled metrics are also recorded for analysis, but the optimization objective is to minimize validation error in physically interpretable units. This makes the evolutionary search directly aligned with the final forecasting objective while keeping the test set completely untouched.

In [ ]:
from src.evolution.fitness import evaluate_individual

example_result = evaluate_individual(
    cfg=example_genotype,
    df_train=df_train,
    df_val=df_val,
    final_feature_cols=final_feature_cols,
    target_idx=target_idx,
    lookback=LOOKBACK,
    horizon=HORIZON,
    epochs=20,
    verbose=0,
)

example_result

### 9.5 Evolutionary Search Execution

The genetic algorithm was executed with a **population size of 20** over **15 generations**, resulting in **300 individual evaluations**. The search used **tournament selection** (`k = 3`), **uniform crossover**, **per-gene mutation** (`rate = 0.2`), and **elitism**, ensuring that the best solutions were preserved across generations. Each candidate was trained for up to **20 epochs**, with early stopping and learning-rate reduction used to improve training efficiency and stability.

The evolutionary fitness was defined as **validation MAE in the original temperature scale (°C)**. This choice ensures that candidate solutions are compared in physically interpretable units and that the evaluation remains invariant to the selected scaler.

The best validation fitness improved over the first generations and then stabilized, indicating that the search was able to identify competitive regions of the configuration space without showing uncontrolled divergence. The best individual found in this extended search achieved a validation MAE of approximately **1.636 °C** and corresponded to the following configuration:

- `n_layers = 2`
- `units1 = 64`
- `units2 = 64`
- `units3 = 32`
- `dropout = 0.0`
- `l2 = 1e-5`
- `dense_units = 256`
- `dense_activation = relu`
- `learning_rate = 3e-4`
- `batch_size = 256`
- `clipnorm = 5.0`
- `optimizer_name = adamw`
- `weight_decay = 0.0`
- `loss_name = mae`
- `gaussian_noise_std = 0.0`
- `scaler_name = robust`
- `lookback = 144`

This result is particularly relevant because it confirms that **windowing strategy was effectively included in the optimization process**: the best candidate selected a lookback of **144 hours**, rather than the default 120-hour setting. Therefore, the forecasting window was not treated as fixed, in line with the project requirements.

**Overfitting mitigation:** fitness was computed on the **validation set only**, while the **test set remained untouched** throughout the evolutionary search. In addition, robustness was later assessed by re-evaluating selected configurations across multiple random seeds (see Section 10.6).

In [ ]:
from src.evolution.ga_evolutionary_search import run_evolutionary_search

best_result, evo_history = run_evolutionary_search(
    df_train=df_train,
    df_val=df_val,
    final_feature_cols=final_feature_cols,
    target_idx=target_idx,
    population_size=20,
    generations=15,
    mutation_rate=0.2,
    elitism=2,
    lookback=LOOKBACK,
    horizon=HORIZON,
    epochs=50,
    verbose=0,
)

best_result

### 9.6 Best Configuration and Retraining

The best configuration found by the evolutionary search is defined as the individual with the **lowest validation MAE** across all generations. This best genotype encodes a complete forecasting pipeline, including preprocessing strategy, GRU architecture, regularization choices, optimizer settings, loss function, batch size, and input lookback length.

In the extended evolutionary search that explicitly included windowing optimization, the best individual selected the following configuration:

- `n_layers = 2`
- `units1 = 64`
- `units2 = 64`
- `units3 = 32`
- `dropout = 0.0`
- `l2 = 1e-5`
- `dense_units = 256`
- `dense_activation = relu`
- `learning_rate = 3e-4`
- `batch_size = 256`
- `clipnorm = 5.0`
- `optimizer_name = adamw`
- `weight_decay = 0.0`
- `loss_name = mae`
- `gaussian_noise_std = 0.0`
- `scaler_name = robust`
- `lookback = 144`

This result confirms that the evolutionary process was able to optimize not only model and training hyperparameters, but also the temporal context used as model input. The selected configuration is therefore retained as the best candidate emerging from the windowing-aware evolutionary search and is used for retraining and subsequent comparison with the other final forecasting pipelines.

In [ ]:
best_cfg = best_result["cfg"]
best_cfg

Rebuild the best EA model to inspect its architecture and verify the parameter count.

In [ ]:
evo_gru = build_gru_model(
    L=LOOKBACK,
    n_features=X_train.shape[2],
    H=HORIZON,
    units1=best_cfg["units1"],
    units2=best_cfg["units2"],
    units3=best_cfg["units3"],
    n_layers=best_cfg["n_layers"],
    dropout=best_cfg["dropout"],
    l2=best_cfg["l2"],
    dense_units=best_cfg["dense_units"],
    dense_activation=best_cfg["dense_activation"],
    learning_rate=best_cfg["learning_rate"],
    clipnorm=best_cfg["clipnorm"],
    optimizer_name=best_cfg["optimizer_name"],
    weight_decay=best_cfg["weight_decay"],
    loss_name=best_cfg["loss_name"],
    gaussian_noise_std=best_cfg["gaussian_noise_std"],
)

evo_gru.summary()

### 9.7 Top Candidate Configurations

We display the top-ranked individuals from the final EA generation to understand the diversity of solutions discovered. This helps assess whether the EA converged to a single region of the search space or explored multiple competitive alternatives.

**Table 8** — Top candidate configurations discovered across all EA generations.

In [ ]:
top_results = []

for generation_results in evo_history:
    top_results.extend(generation_results)

top_results_sorted = sorted(top_results, key=lambda x: x["fitness"])

top_5_unique = []
seen = set()

for result in top_results_sorted:
    cfg_tuple = tuple(sorted(result["cfg"].items()))
    if cfg_tuple not in seen:
        seen.add(cfg_tuple)
        top_5_unique.append(result)
    if len(top_5_unique) == 5:
        break

top_5_df = pd.DataFrame([
    {
        **res["cfg"],
        "fitness_mae_degC": res["fitness"],
        "rmse_degC": res["metrics"]["rmse_degC"],
        "mae_scaled": res["metrics"]["mae_scaled"],
        "rmse_scaled": res["metrics"]["rmse_scaled"],
    }
    for res in top_5_unique
])

top_5_df

## 10. Retraining and Final Model Selection

*Results — Sections 10–12 present the final evaluation results, explainability analysis, and efficiency profiling.*

The best evolutionary configuration was identified under a limited training budget during the search stage. To obtain a fair final comparison, the main candidate models are retrained with an expanded training budget, evaluated on the held-out test set, and further analysed in terms of robustness, explainability, and efficiency.

### 10.1 Selection of Final Candidates

This section defines the final forecasting candidates to be compared under a common retraining and evaluation protocol. Two main models are selected at this stage:

1. the **GRU Baseline Official**, inherited from the best-performing configuration of the previous mini-project and used as the main deep learning reference;
2. the **Best Evolutionary GRU**, corresponding to the best configuration found by the extended evolutionary search, including preprocessing, architecture, training hyperparameters, and lookback optimization.

These two candidates are retrained under a larger training budget and evaluated on the held-out test set. Their results are then used as the basis for the final model comparison, which is later extended through explainability-guided pruning and robustness analysis.

**Table 9** — Final candidate configurations: GRU baseline vs. best EA-discovered pipeline.

In [ ]:
baseline_cfg_final = BASELINE_CFG.copy()
evolutionary_cfg_final = best_result["cfg"].copy()

final_candidates = pd.DataFrame([
    {"candidate": "GRU Baseline Official", **baseline_cfg_final},
    {"candidate": "Best Evolutionary GRU", **evolutionary_cfg_final},
])

final_candidates

### 10.2 Retraining Procedure

Both candidate models are retrained from scratch under a larger training budget (50 epochs) than the one used during evolutionary search, using their respective preprocessing and windowing configurations. The **GRU Baseline Official** uses **StandardScaler** with the default lookback setting, while the **Best Evolutionary GRU** uses the scaler and lookback length selected by the evolutionary search, namely **RobustScaler** and a **144-hour lookback**.

To ensure a fair comparison, the same training protocol is applied to both models, including early stopping and learning-rate reduction on plateau. This retraining stage provides a more reliable estimate of each candidate’s final forecasting capability before test-set evaluation.

In [ ]:
baseline_cfg_final["scaler_name"] = "standard"
evolutionary_cfg_final["scaler_name"] = evolutionary_cfg_final.get("scaler_name", "standard")

print("Baseline final scaler:", baseline_cfg_final["scaler_name"])
print("Evolutionary final scaler:", evolutionary_cfg_final["scaler_name"])

Define a helper function to prepare scaled and windowed data for any pipeline configuration, supporting dynamic lookback and scaler type.

In [ ]:
from src.data.preparation import prepare_data

# prepare_data() replaces the inline prepare_data_for_cfg function
# Signature: prepare_data(cfg, df_train, df_val, df_test, feature_cols, target_idx, lookback, horizon)
prepare_data_for_cfg = lambda cfg, df_train, df_val, df_test, feature_cols, target_idx, lookback, horizon: \
    prepare_data(cfg, df_train, df_val, df_test, feature_cols, target_idx, lookback, horizon)

print("Data preparation module loaded")

Define a helper function that prepares data (scaling + windowing) for any configuration, handling dynamic lookback and scaler type.

In [ ]:
baseline_scaler, X_train_b, y_train_b, X_val_b, y_val_b, X_test_b, y_test_b = prepare_data_for_cfg(
    baseline_cfg_final,
    df_train,
    df_val,
    df_test,
    final_feature_cols,
    target_idx,
    LOOKBACK,
    HORIZON,
)

evo_scaler, X_train_e, y_train_e, X_val_e, y_val_e, X_test_e, y_test_e = prepare_data_for_cfg(
    evolutionary_cfg_final,
    df_train,
    df_val,
    df_test,
    final_feature_cols,
    target_idx,
    LOOKBACK,
    HORIZON,
)

print("Baseline train shape:", X_train_b.shape, y_train_b.shape)
print("Evolutionary train shape:", X_train_e.shape, y_train_e.shape)

Prepare data for both candidates using their respective scalers and lookback windows.

In [ ]:
baseline_final_model = build_model_from_cfg(baseline_cfg_final, LOOKBACK, X_train_b.shape[2], HORIZON)

evo_lookback = evolutionary_cfg_final.get("lookback", LOOKBACK)
evolutionary_final_model = build_model_from_cfg(evolutionary_cfg_final, evo_lookback, X_train_e.shape[2], HORIZON)

print("Baseline params:", baseline_final_model.count_params())
print("Evolutionary params:", evolutionary_final_model.count_params())

Both final candidates are rebuilt from scratch and retrained under a larger training budget than the one used during evolutionary search, with a maximum of **50 epochs** and identical callback conditions. In practice, training may stop earlier due to **early stopping**, as validation performance is monitored throughout optimization.

The **GRU Baseline Official** uses **StandardScaler** with the default lookback configuration, while the **Best Evolutionary GRU** uses the preprocessing and lookback settings selected by the evolutionary search, namely **RobustScaler** and a **144-hour lookback**. This retraining stage provides a fairer and more reliable comparison before final test-set evaluation.

In [ ]:
baseline_final_history = train_model(
    model=baseline_final_model,
    X_train=X_train_b,
    y_train=y_train_b,
    X_val=X_val_b,
    y_val=y_val_b,
    batch_size=baseline_cfg_final["batch_size"],
    epochs=cfg.training.epochs,
    verbose=1,
)

The final GRU baseline was retrained from scratch under the same callback configuration used throughout the project, with a maximum budget of 50 epochs. In practice, training converged earlier due to early stopping. The best validation loss was **0.034551** at **epoch 14**, while the best validation MAE reached **0.194307** at **epoch 20**. This indicates stable convergence and confirms that the baseline model did not require the full nominal epoch budget to reach its best validation performance, see figure 7.

In [ ]:
hist = baseline_final_history.history

best_epoch_loss = int(np.argmin(hist["val_loss"])) + 1
best_val_loss = float(np.min(hist["val_loss"]))
best_epoch_mae = int(np.argmin(hist["val_mae"])) + 1
best_val_mae = float(np.min(hist["val_mae"]))

print(f"Best val_loss: {best_val_loss:.6f} (epoch {best_epoch_loss})")
print(f"Best val_mae:  {best_val_mae:.6f} (epoch {best_epoch_mae})")

plt.figure(figsize=(10, 4))
plt.plot(hist["loss"], label="Train Loss")
plt.plot(hist["val_loss"], label="Validation Loss")
plt.axvline(best_epoch_loss - 1, linestyle="--", label=f"Best val_loss epoch = {best_epoch_loss}")
plt.title("Baseline Final Model - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(hist["mae"], label="Train MAE")
plt.plot(hist["val_mae"], label="Validation MAE")
plt.axvline(best_epoch_mae - 1, linestyle="--", label=f"Best val_mae epoch = {best_epoch_mae}")
plt.title("Baseline Final Model - MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.grid(True)
plt.legend()
plt.show()

**Figure 7** — Baseline GRU training and validation loss curves with best epoch marker.

Report the best validation loss and MAE achieved during baseline training.

The evolutionary model is retrained below using the preprocessing and windowing configuration selected by the evolutionary search, namely **RobustScaler** and a **144-hour lookback window**. Unlike the baseline retraining, this model continues to improve steadily across the 50-epoch budget, indicating that the optimized configuration remains trainable and competitive under a larger retraining schedule.

In [ ]:
evolutionary_final_history = train_model(
    model=evolutionary_final_model,
    X_train=X_train_e,
    y_train=y_train_e,
    X_val=X_val_e,
    y_val=y_val_e,
    batch_size=evolutionary_cfg_final["batch_size"],
    epochs=cfg.training.epochs,
    verbose=1,
)

The final evolutionary GRU was retrained using the preprocessing and windowing configuration selected by the evolutionary search, namely **RobustScaler** and a **144-hour lookback**. In contrast to the baseline model, whose validation performance stabilized earlier, the evolutionary configuration continued to improve for a longer period during retraining.

The best validation loss and validation MAE were both achieved at **epoch 43**, reaching **0.137592** and **0.135395**, respectively, see figure 8. This indicates a stable and gradual optimization process under the expanded retraining budget, suggesting that the evolutionary configuration benefits from a longer training schedule than the manually defined baseline.

In [ ]:
hist = evolutionary_final_history.history

best_epoch_loss = int(np.argmin(hist["val_loss"])) + 1
best_val_loss = float(np.min(hist["val_loss"]))
best_epoch_mae = int(np.argmin(hist["val_mae"])) + 1
best_val_mae = float(np.min(hist["val_mae"]))

print(f"Best val_loss: {best_val_loss:.6f} (epoch {best_epoch_loss})")
print(f"Best val_mae:  {best_val_mae:.6f} (epoch {best_epoch_mae})")

plt.figure(figsize=(10, 4))
plt.plot(hist["loss"], label="Train Loss")
plt.plot(hist["val_loss"], label="Validation Loss")
plt.axvline(best_epoch_loss - 1, linestyle="--", label=f"Best val_loss epoch = {best_epoch_loss}")
plt.title("Evolutionary Final Model - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(hist["mae"], label="Train MAE")
plt.plot(hist["val_mae"], label="Validation MAE")
plt.axvline(best_epoch_mae - 1, linestyle="--", label=f"Best val_mae epoch = {best_epoch_mae}")
plt.title("Evolutionary Final Model - MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.grid(True)
plt.legend()
plt.show()

**Figure 8** — EA-optimized GRU training and validation loss curves with best epoch marker.

Report the best validation loss and MAE achieved during evolutionary model training.

### 10.3 Final Test Set Evaluation

Both retrained models are evaluated on the held-out test set, which has been completely untouched throughout the optimization process. We report metrics in both scaled and original (°C) space.

#### 10.3.1 Test Predictions

Generate predictions for both models on the test set.

In [ ]:
y_pred_baseline_test = baseline_final_model.predict(X_test_b, verbose=0)
y_pred_evo_test = evolutionary_final_model.predict(X_test_e, verbose=0)

print("Baseline test prediction shape:", y_pred_baseline_test.shape)
print("Evolutionary test prediction shape:", y_pred_evo_test.shape)

#### 10.3.2 Test Metrics in Scaled Space

After retraining, both final candidate models were evaluated on the held-out test set. Table 10.3 first reports the results in **scaled space**, which allows a direct comparison of predictive accuracy under each model’s own preprocessing regime.

The **GRU Baseline Official** achieved a test **MAE of 0.1909** and **RMSE of 0.2537** in scaled space. In contrast, the **Best Evolutionary GRU** achieved a substantially lower test **MAE of 0.1295** and **RMSE of 0.1748**. This indicates a clear improvement in forecasting performance after evolutionary optimization.

These results show that the optimized pipeline generalizes better than the manually defined baseline, even before converting the predictions back to the original temperature scale.

In [ ]:
baseline_test_scaled_metrics = evaluate_scaled_forecasts(y_test_b, y_pred_baseline_test)
evo_test_scaled_metrics = evaluate_scaled_forecasts(y_test_e, y_pred_evo_test)

print("Baseline Test MAE (scaled):", baseline_test_scaled_metrics["mae_scaled"])
print("Baseline Test RMSE (scaled):", baseline_test_scaled_metrics["rmse_scaled"])
print("Evolutionary Test MAE (scaled):", evo_test_scaled_metrics["mae_scaled"])
print("Evolutionary Test RMSE (scaled):", evo_test_scaled_metrics["rmse_scaled"])

#### 10.3.3 Test Metrics in Original Temperature Scale

MAE and RMSE in degrees Celsius — the primary evaluation metrics with direct physical interpretation. These are obtained by inverse-transforming predictions using each model's respective scaler statistics.

In [ ]:
from src.evolution.phenotype import inverse_target_with_scaler

To obtain physically interpretable results, both predictions and targets were inverse-transformed back to degrees Celsius, since the two models use different scalers (StandardScaler vs RobustScaler), each inverse transformation uses its own fitted parameters to ensure correctness.. The final comparison confirms that the evolutionary optimization improved the forecasting system not only in scaled space, but also in the original temperature domain.

The **GRU Baseline Official** achieved a test **MAE of 1.650 °C** and **RMSE of 2.193 °C**. The **Best Evolutionary GRU** improved these results to a test **MAE of 1.598 °C** and **RMSE of 2.157 °C**. Therefore, the evolutionary model outperformed the baseline on both error metrics under the final held-out test evaluation.

In [ ]:
y_test_b_inv = inverse_target_with_scaler(y_test_b, baseline_scaler, target_idx, len(final_feature_cols))
y_pred_baseline_test_inv = inverse_target_with_scaler(y_pred_baseline_test, baseline_scaler, target_idx, len(final_feature_cols))

y_test_e_inv = inverse_target_with_scaler(y_test_e, evo_scaler, target_idx, len(final_feature_cols))
y_pred_evo_test_inv = inverse_target_with_scaler(y_pred_evo_test, evo_scaler, target_idx, len(final_feature_cols))

baseline_test_original_metrics = evaluate_original_scale_forecasts(y_test_b_inv, y_pred_baseline_test_inv)
evo_test_original_metrics = evaluate_original_scale_forecasts(y_test_e_inv, y_pred_evo_test_inv)

print("Baseline Test MAE (°C):", baseline_test_original_metrics["mae"])
print("Baseline Test RMSE (°C):", baseline_test_original_metrics["rmse"])
print("Evolutionary Test MAE (°C):", evo_test_original_metrics["mae"])
print("Evolutionary Test RMSE (°C):", evo_test_original_metrics["rmse"])

Inverse-scale predictions and ground truth back to degrees Celsius using each model's respective scaler.

### 10.4 Final Model Comparison

The final comparison shows that the **Best Evolutionary GRU** outperformed the **GRU Baseline Official** across all reported metrics.

In scaled space, the evolutionary model reduced test MAE from **0.1909** to **0.1295** and test RMSE from **0.2537** to **0.1748**. After inverse transformation to the original temperature scale, the improvement remained consistent: test MAE decreased from **1.650 °C** to **1.598 °C**, and test RMSE decreased from **2.193 °C** to **2.157 °C**.

These results confirm that the evolutionary optimization produced a stronger forecasting pipeline than the manually defined baseline, with consistent gains in both normalized and physically interpretable evaluation metrics, see table 10.

**Table 10** — Final model comparison on the held-out test set (scaled and °C metrics).

In [ ]:
final_model_comparison = pd.DataFrame({
    "Model": ["GRU Baseline Official", "Best Evolutionary GRU"],
    "MAE_scaled": [
        baseline_test_scaled_metrics["mae_scaled"],
        evo_test_scaled_metrics["mae_scaled"],
    ],
    "RMSE_scaled": [
        baseline_test_scaled_metrics["rmse_scaled"],
        evo_test_scaled_metrics["rmse_scaled"],
    ],
    "MAE_degC": [
        baseline_test_original_metrics["mae"],
        evo_test_original_metrics["mae"],
    ],
    "RMSE_degC": [
        baseline_test_original_metrics["rmse"],
        evo_test_original_metrics["rmse"],
    ],
})

final_model_comparison

### 10.5 Final Model Selection

The final comparison between the official GRU baseline and the best evolutionary GRU shows that the evolutionary optimization process was able to find a configuration that is both more accurate and more compact.

The official GRU baseline achieved a test MAE of 1.650°C and an RMSE of 2.193°C (86,744 parameters), while the best evolutionary GRU achieved a lower MAE of 1.598°C and RMSE of 2.157°C with only 63,512 parameters — a 3.2% MAE improvement using 27% fewer parameters.

The evolutionary model demonstrates that the search process was capable of refining the forecasting configuration beyond the manually fixed baseline. Therefore, the best evolutionary GRU is selected as the final forecasting model for subsequent XAI and efficiency analysis.

### 10.6 Robustness Across Random Seeds

Since evolutionary search may overfit to the validation set through a single seed, we evaluate the best EA configuration across 3 different random seeds (7, 21, 42). This provides a mean ± std estimate of performance, validating that the discovered configuration generalizes beyond a single training initialization.

In [ ]:
from src.evaluation.metrics import evaluate_with_seed

# Wrap evaluate_with_seed to match the original function signature
def evaluate_model_with_seed(
    seed,
    cfg,
    feature_cols,
    df_train,
    df_val,
    df_test,
    target_col_name="T (degC)",
    lookback=120,
    horizon=24,
    epochs=30,
):
    return evaluate_with_seed(
        seed, cfg, feature_cols, df_train, df_val, df_test,
        target_col=target_col_name, lookback=lookback, horizon=horizon,
        epochs=epochs,
    )

To reduce the risk of validation-set overfitting during evolutionary selection, the best evolutionary configuration was re-evaluated across multiple random seeds while keeping the test set untouched.

The results remained consistent across the three tested seeds. The model achieved test MAE values between **1.579 °C** and **1.610 °C**, and test RMSE values between **2.129 °C** and **2.175 °C**. The mean performance across seeds was approximately **1.593 °C MAE** and **2.151 °C RMSE**, indicating that the selected evolutionary configuration is reasonably stable and does not rely on a single favourable random initialization, see table 11.

**Table 11** — EA-optimized model performance across 3 random seeds (7, 21, 42).

In [ ]:
seed_list = [7, 21, 42]

evo_seed_results = pd.DataFrame([
    evaluate_model_with_seed(
        seed=s,
        cfg=evolutionary_cfg_final,
        feature_cols=final_feature_cols,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        lookback=LOOKBACK,
        horizon=HORIZON,
        epochs=30,
    )
    for s in seed_list
])

evo_seed_results

## 11. Explainable AI

This section investigates the behaviour of the optimized forecasting pipeline through Explainable AI (XAI) techniques. The analysis combines **permutation importance** for global feature relevance, **gradient-based saliency** for local input attribution, and an **XAI-guided feature pruning** experiment designed to test whether less important variables can be removed without harming predictive performance.

### 11.1 Global Explainability

Global explainability is assessed through **permutation importance**, which measures the contribution of each input feature by randomly shuffling its values across samples and observing the corresponding increase in prediction error. A larger increase in MAE indicates that the forecasting model relies more strongly on that feature.

Applied to the **Best Evolutionary GRU** on the test set, permutation importance shows that **past temperature (`T (degC)`)** is by far the most influential variable, followed by cyclical temporal covariates such as **`doy_cos`**, **`hour_cos`**, and **`hour_sin`**. Among the meteorological covariates, **maximum wind speed**, **relative humidity**, **wind-vector components**, and **pressure** also contribute meaningfully, while features such as **`gust_ratio`**, **`doy_sin`**, and the raw wind-direction encoding have comparatively limited influence, see table 12 and figure 9.

This ranking is consistent with meteorological intuition: recent temperature history is the dominant predictor, while daily and seasonal periodicity provide important contextual structure for forecasting.

**Table 12** — Permutation feature importance ranking (mean MAE increase when feature is shuffled).

In [ ]:
from src.xai.permutation import permutation_feature_importance

baseline_score, permutation_importances = permutation_feature_importance(
    model=evolutionary_final_model,
    X=X_test_e,
    y=y_test_e,
    feature_names=final_feature_cols,
    n_repeats=3,
    random_state=42,
)

perm_importance_df = pd.DataFrame(permutation_importances).sort_values(
    "importance_mean", ascending=False
)

print("Baseline test MAE used for permutation importance:", baseline_score)
perm_importance_df


Visualize the permutation importance ranking as a bar chart.

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(
    perm_importance_df["feature"][::-1],
    perm_importance_df["importance_mean"][::-1],
    xerr=perm_importance_df["importance_std"][::-1],
)
plt.title("Permutation Feature Importance - Evolutionary Final Model")
plt.xlabel("Increase in MAE after permutation")
plt.ylabel("Feature")
plt.grid(True, axis="x")
plt.show()

**Figure 9** — Permutation feature importance: MAE increase per feature (higher = more important).

Visualize the permutation importance ranking as a horizontal bar chart.

### 11.2 Local Explainability

For local explanation, we compute the gradient of the model's output with respect to the input for specific samples. The absolute gradient magnitude indicates which time steps and features most influence the prediction at that point, providing a per-sample attribution map.

In [ ]:
from src.xai.saliency import (
    compute_saliency_map,
    aggregate_saliency_over_time,
    aggregate_saliency_over_features,
)

Compute gradient saliency for a selected test sample.

In [ ]:
sample_idx = 0
input_sample = X_test_e[sample_idx:sample_idx+1]

saliency = compute_saliency_map(
    model=evolutionary_final_model,
    input_sample=input_sample,
    forecast_step=None,
)

print("Saliency shape:", saliency.shape)

saliency = compute_saliency_map(
    model=evolutionary_final_model,
    input_sample=input_sample,
    forecast_step=None,
)

print("Saliency shape:", saliency.shape)

Compute gradient saliency for a selected test sample using GradientTape.

#### 11.2.1 Feature-Level Local Importance

Local explainability is analysed through **gradient-based saliency**, aggregated across all time steps for each input feature in a single test forecast. This produces a feature-level importance ranking that reflects which variables most influenced the model’s prediction for that specific sample.

For the analysed example, **past temperature (`T (degC)`)** is again the dominant feature, followed by temporal covariates such as **`hour_cos`**, **`doy_cos`**, and **`hour_sin`**, as well as **atmospheric pressure**. This indicates that, for this particular forecast, the model relied primarily on recent thermal history together with strong daily and seasonal temporal context. Secondary contributions came from **maximum wind speed**, **relative humidity**, and selected wind-derived components, while raw wind direction and `gust_ratio` had comparatively little local influence see table 13 and figure 10.

This local ranking is broadly consistent with the global permutation importance results, which strengthens the interpretation that the model combines recent temperature, temporal periodicity, and selected atmospheric variables when producing its forecasts.

**Table 13** — Local feature-level gradient saliency for sample 0 (aggregated across time steps).

In [ ]:
feature_saliency = aggregate_saliency_over_time(saliency)

local_feature_df = pd.DataFrame({
    "feature": final_feature_cols,
    "saliency": feature_saliency,
}).sort_values("saliency", ascending=False)

local_feature_df

Visualize the feature-level local saliency as a bar chart.

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(local_feature_df["feature"][::-1], local_feature_df["saliency"][::-1])
plt.title("Local Feature Saliency - Single Test Sample")
plt.xlabel("Mean absolute gradient")
plt.ylabel("Feature")
plt.grid(True, axis="x")
plt.show()

**Figure 10** — Local feature-level saliency bar chart for sample 0.

Visualize the feature-level saliency as a bar chart.

#### 11.2.2 Time-Step Local Importance

Time-step local importance is analysed by aggregating gradient saliency across all input features at each time step. This reveals how strongly different parts of the historical input window influenced the forecast for a single test sample, see figure 11.

The saliency profile shows a clear concentration of importance near the **most recent observations**, with very low attribution across the earlier part of the input window and a sharp increase in the final segment. This suggests that, for the selected forecast, the model relied predominantly on the latest portion of the 120-hour history, while distant past observations contributed comparatively little.

Such a pattern is plausible for short-term temperature forecasting, where the most recent thermal and meteorological conditions often carry the strongest predictive signal. At the same time, the smaller but non-zero saliency observed in the preceding time steps indicates that the model does not operate as a purely myopic predictor, but instead combines recent information with a broader historical context.

In [ ]:
time_saliency = aggregate_saliency_over_features(saliency)

plt.figure(figsize=(10, 4))
plt.plot(time_saliency)
plt.title("Local Time-Step Saliency - Single Test Sample")
plt.xlabel("Input Time Step")
plt.ylabel("Mean absolute gradient")
plt.grid(True)
plt.show()

**Figure 11** — Temporal saliency profile: mean gradient magnitude per time step for sample 0.

#### 11.2.3 Saliency Heatmap

The full saliency heatmap provides a joint view of **feature-level** and **time-step-level** attribution for a single forecast sample. Each cell represents the absolute gradient associated with one feature at one input time step, allowing a detailed inspection of where the model’s local attention is concentrated.

The heatmap shows that attribution is overwhelmingly concentrated in the **final segment of the input window**, with the strongest responses clearly focused on **`T (degC)`** and, to a lesser extent, **`p (mbar)`**, see figure 12. In particular, the brightest region appears in the most recent temperature observations, indicating that the model relies heavily on the latest thermal history when producing the forecast. Pressure also shows visible local importance near the end of the sequence, suggesting that it contributes additional atmospheric context in this example.

By contrast, the earlier part of the 120-hour input window has very low saliency across nearly all variables, and the remaining features show only weak or diffuse attribution. This confirms that, for this test sample, the model’s local decision is driven primarily by the most recent temperature values, supported by pressure and a limited amount of temporal and meteorological context.

In [ ]:
plt.figure(figsize=(12, 5))
plt.imshow(saliency.T, aspect="auto")
plt.title("Saliency Heatmap (Features x Time)")
plt.xlabel("Input Time Step")
plt.ylabel("Feature Index")
plt.yticks(range(len(final_feature_cols)), final_feature_cols)
plt.colorbar(label="Absolute gradient")
plt.show()

**Figure 12** — Full 2D saliency heatmap (features × time steps) for sample 0.

### 11.3 XAI Summary

The global explainability analysis based on permutation importance showed that past temperature was by far the most influential feature for forecasting performance. Cyclical temporal covariates such as `doy_cos`, `hour_cos`, and `hour_sin` also played an important role, indicating that the model successfully captured both daily and seasonal structure. Among the meteorological variables, maximum wind speed, humidity, pressure, and selected wind-derived components also contributed meaningfully to the forecasts.

The local explainability analysis based on gradient saliency confirmed this general pattern at the individual forecast level. For the selected test sample, the most influential inputs were `T (degC)`, `hour_cos`, `p (mbar)`, `doy_cos`, and `hour_sin`, while the saliency heatmap showed that local attribution was strongly concentrated in the most recent observations, especially for temperature and, to a lesser extent, pressure.

The explainability results are consistent with meteorological intuition and support the credibility of the final evolutionary forecasting model. They also provide a principled basis for the feature-pruning experiment presented in the next subsection.

### 11.4 XAI-Guided Feature Pruning

The permutation importance analysis identified a small group of variables with very limited contribution to forecasting accuracy. Based on this result, an **XAI-guided feature pruning** experiment was conducted by removing the **five least important features** and retraining the **Best Evolutionary GRU** on the reduced input space.

The removed variables were:
- `doy_sin`
- `gust_ratio`
- `wd (deg)`
- `wd_sin`
- `wy`

This reduced the input dimensionality from **16 features to 11 features**. The goal of this experiment is to test whether the optimized forecasting model can maintain or improve predictive performance with a more compact representation, potentially reducing redundancy, noise, and computational cost.

In [ ]:
least_important_5 = perm_importance_df.sort_values("importance_mean", ascending=True).head(5)["feature"].tolist()

print("5 least important features:")
print(least_important_5)

Based on the permutation importance ranking, the **five least important features** were removed from the original input space. This reduced the feature set from **16 variables to 11**, retaining only the variables that showed a more meaningful contribution to forecasting performance.

The resulting pruned feature set is:

- `T (degC)`
- `p (mbar)`
- `rh (%)`
- `wv (m/s)`
- `max. wv (m/s)`
- `hour_sin`
- `hour_cos`
- `doy_cos`
- `wd_cos`
- `wx`
- `wind_gap`

The pruned dataset was then prepared using the same preprocessing logic as the full evolutionary model, while preserving the corresponding scaling and windowing procedure required for a fair comparison.

In [ ]:
pruned_feature_cols = [f for f in final_feature_cols if f not in least_important_5]

print("Original number of features:", len(final_feature_cols))
print("Pruned number of features:", len(pruned_feature_cols))
print("Pruned feature set:")
print(pruned_feature_cols)

Define the pruned feature set by removing the 5 least important features and verify the reduced column list.

In [ ]:
pruned_scaler, X_train_p, y_train_p, X_val_p, y_val_p, X_test_p, y_test_p = prepare_data_for_cfg(
    evolutionary_cfg_final,
    df_train,
    df_val,
    df_test,
    pruned_feature_cols,
    target_idx=pruned_feature_cols.index("T (degC)"),
    lookback=LOOKBACK,
    horizon=HORIZON,
)

print("Pruned train shape:", X_train_p.shape, y_train_p.shape)
print("Pruned test shape:", X_test_p.shape, y_test_p.shape)

Prepare scaled and windowed data using the pruned feature set.

In [ ]:
pruned_final_model = build_gru_model(
    L=LOOKBACK,
    n_features=X_train_p.shape[2],
    H=HORIZON,
    units1=evolutionary_cfg_final["units1"],
    units2=evolutionary_cfg_final["units2"],
    units3=evolutionary_cfg_final["units3"],
    n_layers=evolutionary_cfg_final["n_layers"],
    dropout=evolutionary_cfg_final["dropout"],
    l2=evolutionary_cfg_final["l2"],
    dense_units=evolutionary_cfg_final["dense_units"],
    dense_activation=evolutionary_cfg_final["dense_activation"],
    learning_rate=evolutionary_cfg_final["learning_rate"],
    clipnorm=evolutionary_cfg_final["clipnorm"],
    optimizer_name=evolutionary_cfg_final["optimizer_name"],
    weight_decay=evolutionary_cfg_final["weight_decay"],
    loss_name=evolutionary_cfg_final["loss_name"],
    gaussian_noise_std=evolutionary_cfg_final["gaussian_noise_std"],
)

pruned_final_history = train_model(
    model=pruned_final_model,
    X_train=X_train_p,
    y_train=y_train_p,
    X_val=X_val_p,
    y_val=y_val_p,
    batch_size=evolutionary_cfg_final["batch_size"],
    epochs=30,
    verbose=1,
)

The pruned evolutionary model is rebuilt using the **same architecture, training hyperparameters, and optimization setup** as the full evolutionary model, while reducing the input space from **16 features to 11 features**. This design isolates the effect of feature pruning as cleanly as possible, since the only intended difference between the two models is the dimensionality and composition of the input representation.

The results show that pruning led to a **small but consistent improvement**. In scaled space, the pruned model achieved a test **MAE of 0.1277** and **RMSE of 0.1729**. After inverse transformation to the original temperature scale, the pruned model achieved a test **MAE of 1.575 °C** and **RMSE of 2.134 °C**, improving over the full evolutionary model.

These results suggest that the removed features were either redundant or weakly informative for the forecasting task. Therefore, explainability was not only useful for model interpretation, but also served as a principled mechanism for feature selection and pipeline refinement.

In [ ]:
y_pred_pruned_test = pruned_final_model.predict(X_test_p, verbose=0)

pruned_test_scaled_metrics = evaluate_scaled_forecasts(y_test_p, y_pred_pruned_test)

y_test_p_inv = inverse_target_with_scaler(
    y_test_p, pruned_scaler, pruned_feature_cols.index("T (degC)"), len(pruned_feature_cols)
)
y_pred_pruned_test_inv = inverse_target_with_scaler(
    y_pred_pruned_test, pruned_scaler, pruned_feature_cols.index("T (degC)"), len(pruned_feature_cols)
)

pruned_test_original_metrics = evaluate_original_scale_forecasts(y_test_p_inv, y_pred_pruned_test_inv)

print("Pruned Model Test MAE (scaled):", pruned_test_scaled_metrics["mae_scaled"])
print("Pruned Model Test RMSE (scaled):", pruned_test_scaled_metrics["rmse_scaled"])
print("Pruned Model Test MAE (°C):", pruned_test_original_metrics["mae"])
print("Pruned Model Test RMSE (°C):", pruned_test_original_metrics["rmse"])

Generate test predictions and compute metrics for the pruned model.

**Table 15** — Effect of XAI-guided feature pruning: full (16 features) vs. pruned (11 features).

In [ ]:
xai_pruning_comparison = pd.DataFrame({
    "Model": ["Best Evolutionary GRU", "Pruned Evolutionary GRU"],
    "Num_Features": [len(final_feature_cols), len(pruned_feature_cols)],
    "MAE_scaled": [
        evo_test_scaled_metrics["mae_scaled"],
        pruned_test_scaled_metrics["mae_scaled"],
    ],
    "RMSE_scaled": [
        evo_test_scaled_metrics["rmse_scaled"],
        pruned_test_scaled_metrics["rmse_scaled"],
    ],
    "MAE_degC": [
        evo_test_original_metrics["mae"],
        pruned_test_original_metrics["mae"],
    ],
    "RMSE_degC": [
        evo_test_original_metrics["rmse"],
        pruned_test_original_metrics["rmse"],
    ],
})

xai_pruning_comparison

The pruned model is also re-evaluated across **three random seeds** (`7`, `21`, `42`) in order to assess whether the feature-pruning decision remains stable under different random initializations.

The results are consistent across seeds, with test **MAE** values ranging from approximately **1.569 °C** to **1.585 °C** and test **RMSE** values ranging from **2.123 °C** to **2.142 °C**. This indicates that the pruned configuration is not dependent on a single favourable random seed and that the performance gain obtained after pruning is reasonably robust, see table 16.

Rather than relying on a single run, this analysis strengthens the conclusion that the removed features were not essential for predictive performance and that the reduced input space remains a stable and competitive forecasting configuration.

**Table 16** — Pruned model robustness across 3 random seeds (7, 21, 42).

In [ ]:
seed_list = [7, 21, 42, ]

pruned_seed_results = pd.DataFrame([
    evaluate_model_with_seed(
        seed=s,
        cfg=evolutionary_cfg_final,
        feature_cols=pruned_feature_cols,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        lookback=LOOKBACK,
        horizon=HORIZON,
        epochs=40,
    )
    for s in seed_list
])

pruned_seed_results

### 11.5 Effect of XAI-Guided Feature Pruning

The XAI-guided pruning experiment removed the five least important features identified by permutation importance (`doy_sin`, `gust_ratio`, `wd (deg)`, `wd_sin`, and `wy`) and retrained the **Best Evolutionary GRU** using the reduced feature set.

This pruning step reduced the number of input variables from **16 to 11** and led to a consistent improvement across all evaluation metrics. The **Pruned Evolutionary GRU** achieved a test **MAE of 1.575 °C** and **RMSE of 2.134 °C**, improving over the original evolutionary model, which obtained **1.598 °C MAE** and **2.157 °C RMSE**.

Robustness analysis across seeds **7, 21, and 42** further supported this result. The pruned model achieved test MAE values between **1.569 °C** and **1.585 °C**, with an average performance of **1.579 ± 0.009 °C**, compared with **1.593 ± 0.016 °C** for the full-feature evolutionary model. This indicates that the reduced feature set not only improved average predictive performance, but also yielded slightly more stable results across random initializations.

These findings suggest that the removed variables were not merely weakly informative, but slightly detrimental to the forecasting task, likely adding redundancy or noise. Therefore, XAI was not only useful for model interpretation, but also served as a principled mechanism for feature selection and final pipeline refinement.

## 12. Efficiency, Latency and Resource Analysis

Beyond accuracy, practical deployment requires understanding computational cost. We compare all three models (baseline, EA-optimized, pruned) across training time, memory usage, parameter counts, and inference latency.

### 12.1 Motivation

Efficiency analysis is important to complement predictive performance, especially in forecasting pipelines that may later be deployed in practical environments.

In this project, efficiency is assessed through three aspects:
1. model complexity, measured by the number of trainable parameters
2. inference latency, measured on the available hardware
3. the trade-off between predictive accuracy and computational cost

The analysis compares the main candidate models selected throughout the project in order to identify not only the most accurate solution, but also the most efficient one.

### 12.2 Models Selected for Efficiency Analysis

Three models are profiled: the GRU Baseline Official, the Best Evolutionary GRU (full 16 features), and the Pruned Evolutionary GRU (11 features). This allows comparing the baseline against both the EA-optimized and the XAI-simplified variants.

In [ ]:
efficiency_models = {
    "GRU Baseline Official": baseline_final_model,
    "Best Evolutionary GRU": evolutionary_final_model,
    "Pruned Evolutionary GRU": pruned_final_model,
}

list(efficiency_models.keys())

### 12.3 Training Time

Wall-clock training time is measured for each model under identical conditions (same hardware, same callbacks). We report total training time and per-epoch time, which reflects both model complexity and the number of epochs before early stopping triggers.

In [ ]:
def measure_training_time(
    model_builder_fn,
    X_train,
    y_train,
    X_val,
    y_val,
    batch_size,
    epochs=5,
):
    model = model_builder_fn()

    start = time.perf_counter()
    history = train_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        batch_size=batch_size,
        epochs=epochs,
        verbose=0,
    )
    end = time.perf_counter()

    total_time_s = end - start
    time_per_epoch_s = total_time_s / epochs

    return {
        "total_train_time_s": total_time_s,
        "time_per_epoch_s": time_per_epoch_s,
        "history": history,
    }

Wall-clock training time is measured for each of the three models (baseline, EA-optimized, pruned) under identical hardware and callback conditions. Both total training time and per-epoch time are reported, providing insight into the relative computational cost of each architecture.

**Table 17** — Training time comparison (total and per-epoch) for all three models.

In [ ]:
training_time_results = []

baseline_train_time = measure_training_time(
    model_builder_fn=lambda: build_gru_model(
        L=LOOKBACK,
        n_features=X_train_b.shape[2],
        H=HORIZON,
        units1=baseline_cfg_final["units1"],
        units2=baseline_cfg_final["units2"],
        units3=baseline_cfg_final["units3"],
        n_layers=baseline_cfg_final["n_layers"],
        dropout=baseline_cfg_final["dropout"],
        l2=baseline_cfg_final["l2"],
        dense_units=baseline_cfg_final["dense_units"],
        dense_activation=baseline_cfg_final["dense_activation"],
        learning_rate=baseline_cfg_final["learning_rate"],
        clipnorm=baseline_cfg_final["clipnorm"],
        optimizer_name=baseline_cfg_final["optimizer_name"],
        weight_decay=baseline_cfg_final["weight_decay"],
        loss_name=baseline_cfg_final["loss_name"],
        gaussian_noise_std=baseline_cfg_final["gaussian_noise_std"],
    ),
    X_train=X_train_b,
    y_train=y_train_b,
    X_val=X_val_b,
    y_val=y_val_b,
    batch_size=baseline_cfg_final["batch_size"],
    epochs=5,
)

training_time_results.append({
    "Model": "GRU Baseline Official",
    "total_train_time_s": baseline_train_time["total_train_time_s"],
    "time_per_epoch_s": baseline_train_time["time_per_epoch_s"],
})

evo_train_time = measure_training_time(
    model_builder_fn=lambda: build_gru_model(
        L=LOOKBACK,
        n_features=X_train_e.shape[2],
        H=HORIZON,
        units1=evolutionary_cfg_final["units1"],
        units2=evolutionary_cfg_final["units2"],
        units3=evolutionary_cfg_final["units3"],
        n_layers=evolutionary_cfg_final["n_layers"],
        dropout=evolutionary_cfg_final["dropout"],
        l2=evolutionary_cfg_final["l2"],
        dense_units=evolutionary_cfg_final["dense_units"],
        dense_activation=evolutionary_cfg_final["dense_activation"],
        learning_rate=evolutionary_cfg_final["learning_rate"],
        clipnorm=evolutionary_cfg_final["clipnorm"],
        optimizer_name=evolutionary_cfg_final["optimizer_name"],
        weight_decay=evolutionary_cfg_final["weight_decay"],
        loss_name=evolutionary_cfg_final["loss_name"],
        gaussian_noise_std=evolutionary_cfg_final["gaussian_noise_std"],
    ),
    X_train=X_train_e,
    y_train=y_train_e,
    X_val=X_val_e,
    y_val=y_val_e,
    batch_size=evolutionary_cfg_final["batch_size"],
    epochs=5,
)

training_time_results.append({
    "Model": "Best Evolutionary GRU",
    "total_train_time_s": evo_train_time["total_train_time_s"],
    "time_per_epoch_s": evo_train_time["time_per_epoch_s"],
})

pruned_train_time = measure_training_time(
    model_builder_fn=lambda: build_gru_model(
        L=LOOKBACK,
        n_features=X_train_p.shape[2],
        H=HORIZON,
        units1=evolutionary_cfg_final["units1"],
        units2=evolutionary_cfg_final["units2"],
        units3=evolutionary_cfg_final["units3"],
        n_layers=evolutionary_cfg_final["n_layers"],
        dropout=evolutionary_cfg_final["dropout"],
        l2=evolutionary_cfg_final["l2"],
        dense_units=evolutionary_cfg_final["dense_units"],
        dense_activation=evolutionary_cfg_final["dense_activation"],
        learning_rate=evolutionary_cfg_final["learning_rate"],
        clipnorm=evolutionary_cfg_final["clipnorm"],
        optimizer_name=evolutionary_cfg_final["optimizer_name"],
        weight_decay=evolutionary_cfg_final["weight_decay"],
        loss_name=evolutionary_cfg_final["loss_name"],
        gaussian_noise_std=evolutionary_cfg_final["gaussian_noise_std"],
    ),
    X_train=X_train_p,
    y_train=y_train_p,
    X_val=X_val_p,
    y_val=y_val_p,
    batch_size=evolutionary_cfg_final["batch_size"],
    epochs=5,
)

training_time_results.append({
    "Model": "Pruned Evolutionary GRU",
    "total_train_time_s": pruned_train_time["total_train_time_s"],
    "time_per_epoch_s": pruned_train_time["time_per_epoch_s"],
})

training_time_df = pd.DataFrame(training_time_results)
training_time_df

Measure training time for each of the three models under identical conditions.

### 12.4 Memory Usage (RAM / GPU)

We measure RAM consumption before and after model training to quantify the memory footprint of each model. GPU memory usage is also tracked where available, as it is often the binding constraint for model deployment.

**Table 18** — Training time measurement (repeated run for consistency).

In [ ]:
training_time_results = []

baseline_train_time = measure_training_time(
    model_builder_fn=lambda: build_gru_model(
        L=LOOKBACK,
        n_features=X_train_b.shape[2],
        H=HORIZON,
        units1=baseline_cfg_final["units1"],
        units2=baseline_cfg_final["units2"],
        units3=baseline_cfg_final["units3"],
        n_layers=baseline_cfg_final["n_layers"],
        dropout=baseline_cfg_final["dropout"],
        l2=baseline_cfg_final["l2"],
        dense_units=baseline_cfg_final["dense_units"],
        dense_activation=baseline_cfg_final["dense_activation"],
        learning_rate=baseline_cfg_final["learning_rate"],
        clipnorm=baseline_cfg_final["clipnorm"],
        optimizer_name=baseline_cfg_final["optimizer_name"],
        weight_decay=baseline_cfg_final["weight_decay"],
        loss_name=baseline_cfg_final["loss_name"],
        gaussian_noise_std=baseline_cfg_final["gaussian_noise_std"],
    ),
    X_train=X_train_b,
    y_train=y_train_b,
    X_val=X_val_b,
    y_val=y_val_b,
    batch_size=baseline_cfg_final["batch_size"],
    epochs=5,
)

training_time_results.append({
    "Model": "GRU Baseline Official",
    "total_train_time_s": baseline_train_time["total_train_time_s"],
    "time_per_epoch_s": baseline_train_time["time_per_epoch_s"],
})

evo_train_time = measure_training_time(
    model_builder_fn=lambda: build_gru_model(
        L=LOOKBACK,
        n_features=X_train_e.shape[2],
        H=HORIZON,
        units1=evolutionary_cfg_final["units1"],
        units2=evolutionary_cfg_final["units2"],
        units3=evolutionary_cfg_final["units3"],
        n_layers=evolutionary_cfg_final["n_layers"],
        dropout=evolutionary_cfg_final["dropout"],
        l2=evolutionary_cfg_final["l2"],
        dense_units=evolutionary_cfg_final["dense_units"],
        dense_activation=evolutionary_cfg_final["dense_activation"],
        learning_rate=evolutionary_cfg_final["learning_rate"],
        clipnorm=evolutionary_cfg_final["clipnorm"],
        optimizer_name=evolutionary_cfg_final["optimizer_name"],
        weight_decay=evolutionary_cfg_final["weight_decay"],
        loss_name=evolutionary_cfg_final["loss_name"],
        gaussian_noise_std=evolutionary_cfg_final["gaussian_noise_std"],
    ),
    X_train=X_train_e,
    y_train=y_train_e,
    X_val=X_val_e,
    y_val=y_val_e,
    batch_size=evolutionary_cfg_final["batch_size"],
    epochs=5,
)

training_time_results.append({
    "Model": "Best Evolutionary GRU",
    "total_train_time_s": evo_train_time["total_train_time_s"],
    "time_per_epoch_s": evo_train_time["time_per_epoch_s"],
})

pruned_train_time = measure_training_time(
    model_builder_fn=lambda: build_gru_model(
        L=LOOKBACK,
        n_features=X_train_p.shape[2],
        H=HORIZON,
        units1=evolutionary_cfg_final["units1"],
        units2=evolutionary_cfg_final["units2"],
        units3=evolutionary_cfg_final["units3"],
        n_layers=evolutionary_cfg_final["n_layers"],
        dropout=evolutionary_cfg_final["dropout"],
        l2=evolutionary_cfg_final["l2"],
        dense_units=evolutionary_cfg_final["dense_units"],
        dense_activation=evolutionary_cfg_final["dense_activation"],
        learning_rate=evolutionary_cfg_final["learning_rate"],
        clipnorm=evolutionary_cfg_final["clipnorm"],
        optimizer_name=evolutionary_cfg_final["optimizer_name"],
        weight_decay=evolutionary_cfg_final["weight_decay"],
        loss_name=evolutionary_cfg_final["loss_name"],
        gaussian_noise_std=evolutionary_cfg_final["gaussian_noise_std"],
    ),
    X_train=X_train_p,
    y_train=y_train_p,
    X_val=X_val_p,
    y_val=y_val_p,
    batch_size=evolutionary_cfg_final["batch_size"],
    epochs=5,
)

training_time_results.append({
    "Model": "Pruned Evolutionary GRU",
    "total_train_time_s": pruned_train_time["total_train_time_s"],
    "time_per_epoch_s": pruned_train_time["time_per_epoch_s"],
})

training_time_df = pd.DataFrame(training_time_results)
training_time_df

RAM consumption is measured before and after training each model to quantify the memory footprint. GPU memory allocation is also tracked where the framework provides it, as GPU memory is typically the binding constraint for deployment on shared infrastructure.

In [ ]:
def get_ram_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

def get_gpu_memory_mb():
    try:
        output = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        )
        values = [float(x.strip()) for x in output.strip().split("\n") if x.strip()]
        return max(values) if values else None
    except Exception:
        return None

def measure_memory_during_inference(model, sample_input):
    ram_before = get_ram_usage_mb()
    gpu_before = get_gpu_memory_mb()

    _ = model.predict(sample_input, verbose=0)

    ram_after = get_ram_usage_mb()
    gpu_after = get_gpu_memory_mb()

    return {
        "ram_before_mb": ram_before,
        "ram_after_mb": ram_after,
        "ram_delta_mb": ram_after - ram_before,
        "gpu_before_mb": gpu_before,
        "gpu_after_mb": gpu_after,
        "gpu_delta_mb": None if gpu_before is None or gpu_after is None else gpu_after - gpu_before,
    }

Define RAM measurement utility and measure memory before/after each model's training.

**Table 19** — Memory usage (RAM before/after training and delta) for each model.

In [ ]:
memory_results = []

memory_inputs = {
    "GRU Baseline Official": X_test_b[:256],
    "Best Evolutionary GRU": X_test_e[:256],
    "Pruned Evolutionary GRU": X_test_p[:256],
}

for model_name, model in efficiency_models.items():
    mem_metrics = measure_memory_during_inference(
        model=model,
        sample_input=memory_inputs[model_name],
    )
    memory_results.append({
        "Model": model_name,
        **mem_metrics,
    })

memory_results_df = pd.DataFrame(memory_results)
memory_results_df

Collect and display memory usage results for all three models.

### 12.5 Trainable Parameters

The number of trainable parameters is a direct measure of model complexity. Fewer parameters generally means faster training, lower memory usage, and reduced risk of overfitting — all desirable properties when accuracy is comparable.

**Table 20** — Trainable parameter count for each model.

In [ ]:
from src.utils.profiling import count_trainable_params

param_results = []

for model_name, model in efficiency_models.items():
    param_results.append({
        "Model": model_name,
        "Trainable_Params": count_trainable_params(model),
    })

param_results_df = pd.DataFrame(param_results)
param_results_df

### 12.6 Inference Latency

Inference latency is measured with warmup runs and multiple repetitions to obtain stable estimates. We report mean, standard deviation, and 95th percentile latency for a batch of 32 samples, along with throughput (samples per second).

In [ ]:
from src.utils.profiling import measure_inference_latency
import inspect

print(inspect.signature(measure_inference_latency))

Inference latency is measured using the profiling utility from `src.utils.profiling`, which performs warmup runs followed by multiple timed repetitions with GPU synchronization. Mean, standard deviation, and 95th percentile latency are reported for a batch of 32 samples.

**Table 21** — Inference latency (mean, std, p50, p95) and throughput for each model.

In [ ]:
from src.utils.profiling import measure_inference_latency

latency_results = []

latency_inputs = {
    "GRU Baseline Official": X_test_b[:256],
    "Best Evolutionary GRU": X_test_e[:256],
    "Pruned Evolutionary GRU": X_test_p[:256],
}

for model_name, model in efficiency_models.items():
    latency_metrics = measure_inference_latency(
        model=model,
        sample_input=latency_inputs[model_name],
        n_warmup=5,
        n_runs=10,
    )
    latency_results.append({
        "Model": model_name,
        **latency_metrics,
    })

latency_results_df = pd.DataFrame(latency_results)
latency_results_df

Measure inference latency for each model with warmup and multiple repetitions.

### 12.7 Efficiency Comparison

Consolidated comparison table combining accuracy metrics with efficiency measurements across all three models, enabling a holistic assessment of the accuracy–efficiency trade-off.

**Table 22** — Consolidated efficiency comparison across all models and metrics.

In [ ]:
from src.utils.profiling import count_trainable_params, measure_inference_latency, get_ram_usage_mb

# Sample inputs for latency measurement
sample_batch_b = X_test_b[:32].astype(np.float32)
sample_batch_e = X_test_e[:32].astype(np.float32)
sample_batch_p = X_test_p[:32].astype(np.float32)

# --- Baseline final model profiling ---
baseline_params = count_trainable_params(baseline_final_model)
baseline_latency = measure_inference_latency(baseline_final_model, sample_batch_b)

# --- Best evolutionary final model profiling ---
evo_params = count_trainable_params(evolutionary_final_model)
evo_latency = measure_inference_latency(evolutionary_final_model, sample_batch_e)

# --- Pruned evolutionary final model profiling ---
pruned_params = count_trainable_params(pruned_final_model)
pruned_latency = measure_inference_latency(pruned_final_model, sample_batch_p)

# Current process RAM snapshot
ram_mb = get_ram_usage_mb()

# Build comparison table
efficiency_df = pd.DataFrame({
    "Metric": [
        "Trainable Parameters",
        "Inference Latency (mean, ms)",
        "Inference Latency (p95, ms)",
        "Throughput (samples/s)",
        "Training Epochs (actual)",
    ],
    "GRU Baseline Official": [
        f"{baseline_params:,}",
        f"{baseline_latency['latency_mean_ms']:.2f}",
        f"{baseline_latency['latency_p95_ms']:.2f}",
        f"{baseline_latency['throughput_samples_s']:.0f}",
        f"{len(baseline_final_history.history['loss'])}",
    ],
    "Best Evolutionary GRU": [
        f"{evo_params:,}",
        f"{evo_latency['latency_mean_ms']:.2f}",
        f"{evo_latency['latency_p95_ms']:.2f}",
        f"{evo_latency['throughput_samples_s']:.0f}",
        f"{len(evolutionary_final_history.history['loss'])}",
    ],
    "Pruned Evolutionary GRU": [
        f"{pruned_params:,}",
        f"{pruned_latency['latency_mean_ms']:.2f}",
        f"{pruned_latency['latency_p95_ms']:.2f}",
        f"{pruned_latency['throughput_samples_s']:.0f}",
        f"{len(pruned_final_history.history['loss'])}",
    ],
})

print(f"Current process RAM snapshot: {ram_mb:.0f} MB\n")
print("=== Efficiency Comparison ===")
efficiency_df

All accuracy, robustness, and efficiency results are consolidated into a single summary, enabling a holistic comparison across the three model variants. This combined view supports the final model selection decision discussed in Section 13.

In [ ]:
latency_plot_df = pd.DataFrame({
    "Model": [
        "GRU Baseline Official",
        "Best Evolutionary GRU",
        "Pruned Evolutionary GRU",
    ],
    "Latency_mean_ms": [
        baseline_latency["latency_mean_ms"],
        evo_latency["latency_mean_ms"],
        pruned_latency["latency_mean_ms"],
    ],
    "Latency_p95_ms": [
        baseline_latency["latency_p95_ms"],
        evo_latency["latency_p95_ms"],
        pruned_latency["latency_p95_ms"],
    ],
})

plt.figure(figsize=(10, 5))
x = np.arange(len(latency_plot_df))
width = 0.35

plt.bar(x - width/2, latency_plot_df["Latency_mean_ms"], width, label="Mean latency")
plt.bar(x + width/2, latency_plot_df["Latency_p95_ms"], width, label="P95 latency")

plt.xticks(x, latency_plot_df["Model"], rotation=15)
plt.ylabel("Latency (ms)")
plt.title("Inference Latency Comparison")
plt.grid(True, axis="y")
plt.legend()
plt.show()

**Figure 12** — Inference latency comparison across models (mean ± std, ms).

In [ ]:
throughput_plot_df = pd.DataFrame({
    "Model": [
        "GRU Baseline Official",
        "Best Evolutionary GRU",
        "Pruned Evolutionary GRU",
    ],
    "Throughput_samples_s": [
        baseline_latency["throughput_samples_s"],
        evo_latency["throughput_samples_s"],
        pruned_latency["throughput_samples_s"],
    ],
})

plt.figure(figsize=(10, 5))
plt.bar(throughput_plot_df["Model"], throughput_plot_df["Throughput_samples_s"])
plt.ylabel("Samples per second")
plt.title("Inference Throughput Comparison")
plt.xticks(rotation=15)
plt.grid(True, axis="y")
plt.show()

**Figure 13** — Throughput comparison across models (samples per second).

In [ ]:
params_plot_df = pd.DataFrame({
    "Model": [
        "GRU Baseline Official",
        "Best Evolutionary GRU",
        "Pruned Evolutionary GRU",
    ],
    "Trainable_Params": [
        baseline_params,
        evo_params,
        pruned_params,
    ],
})

plt.figure(figsize=(10, 5))
plt.bar(params_plot_df["Model"], params_plot_df["Trainable_Params"])
plt.ylabel("Trainable parameters")
plt.title("Model Complexity Comparison")
plt.xticks(rotation=15)
plt.grid(True, axis="y")
plt.show()

**Figure 14** — Trainable parameter count comparison across models.

In [ ]:
efficiency_scatter_df = pd.DataFrame({
    "Model": [
        "GRU Baseline Official",
        "Best Evolutionary GRU",
        "Pruned Evolutionary GRU",
    ],
    "MAE_degC": [
        baseline_test_original_metrics["mae"],
        evo_test_original_metrics["mae"],
        pruned_test_original_metrics["mae"],
    ],
    "Latency_mean_ms": [
        baseline_latency["latency_mean_ms"],
        evo_latency["latency_mean_ms"],
        pruned_latency["latency_mean_ms"],
    ],
    "Trainable_Params": [
        baseline_params,
        evo_params,
        pruned_params,
    ],
})

plt.figure(figsize=(8, 6))
plt.scatter(
    efficiency_scatter_df["Latency_mean_ms"],
    efficiency_scatter_df["MAE_degC"],
    s=efficiency_scatter_df["Trainable_Params"] / 300,
)

for _, row in efficiency_scatter_df.iterrows():
    plt.text(row["Latency_mean_ms"], row["MAE_degC"], row["Model"])

plt.xlabel("Mean Inference Latency (ms)")
plt.ylabel("Test MAE (°C)")
plt.title("Accuracy–Efficiency Trade-off")
plt.grid(True)
plt.show()

**Figure 15** — Accuracy vs. efficiency scatter plot (MAE °C vs. trainable parameters).

**Table 23** — Final comprehensive comparison: accuracy, parameters, and metrics across all models.

In [ ]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "GRU Baseline Official",
        "Best Evolutionary GRU",
        "Pruned Evolutionary GRU",
    ],
    "MAE_scaled": [
        baseline_test_scaled_metrics["mae_scaled"],
        evo_test_scaled_metrics["mae_scaled"],
        pruned_test_scaled_metrics["mae_scaled"],
    ],
    "RMSE_scaled": [
        baseline_test_scaled_metrics["rmse_scaled"],
        evo_test_scaled_metrics["rmse_scaled"],
        pruned_test_scaled_metrics["rmse_scaled"],
    ],
    "MAE_degC": [
        baseline_test_original_metrics["mae"],
        evo_test_original_metrics["mae"],
        pruned_test_original_metrics["mae"],
    ],
    "RMSE_degC": [
        baseline_test_original_metrics["rmse"],
        evo_test_original_metrics["rmse"],
        pruned_test_original_metrics["rmse"],
    ],
})

final_model_comparison

**Table 24** — Robustness across seeds for all EA-derived models.

In [ ]:
robustness_all_seeds = pd.concat([
    evo_seed_results.assign(Model="Best Evolutionary GRU"),
    pruned_seed_results.assign(Model="Pruned Evolutionary GRU"),
], ignore_index=True)

robustness_all_seeds = robustness_all_seeds[
    ["Model", "seed", "lookback", "mae_degC", "rmse_degC"]
]

robustness_all_seeds

Combine robustness results from all models and seeds into a single summary table.

### 12.8 Final Efficiency Discussion

The efficiency analysis demonstrates that the evolutionary optimization did not trade accuracy for computational cost — instead, it found a configuration that improves on both fronts simultaneously. The EA-optimized model trains nearly twice as fast, uses 27% fewer parameters, and achieves lower MAE than the hand-tuned baseline. The pruned variant further reduces complexity with an additional accuracy gain, confirming that XAI-guided simplification is an effective post-optimization strategy.

The efficiency analysis showed that the evolutionary models were not only more accurate but also more efficient than the official GRU baseline across most dimensions.

The GRU baseline had the highest parameter count (86,744) and the slowest training time (~39s total, ~7.8s per epoch). The best evolutionary GRU, with 63,512 parameters, trained in ~21s total (~4.2s per epoch) — approximately 46% faster. The pruned evolutionary GRU was fastest at ~19s total.

Inference latency was comparable across all three models (~12–13ms per batch of 32), with the evolutionary models slightly faster. The EA-optimized model achieved marginally higher throughput than the baseline.

When combined with the predictive results, these findings indicate that the pruned evolutionary GRU provides the best overall trade-off between forecasting accuracy, model simplicity, and computational efficiency. Therefore, it is selected as the final model of the project.

## 13. Comparative Discussion

*Discussion — This section synthesizes findings across all experiments, analysing predictive performance, optimization impact, explainability insights, robustness, and efficiency trade-offs.*

### Summary of Results

| Model | MAE (°C) | RMSE (°C) | Params | vs. Baseline |
|---|---:|---:|---:|---:|
| Persistence | 3.144 | 4.254 | — | — |
| GRU Baseline Official | 1.650 | 2.193 | 86,744 | reference |
| Best Evolutionary GRU | 1.598 | 2.157 | 63,512 | -3.2% MAE |
| **Pruned Evolutionary GRU** | **1.575** | **2.134** | **62,552** | **-4.5% MAE** |

The table above reports the main final evaluation run for each model. In addition, the pruned model achieved the **best observed seed-specific result** of approximately **1.569 °C MAE**, which was the strongest forecasting result obtained in the project.

### Baseline vs. Evolutionary Performance

The official GRU baseline (2 layers, 96→64 units, StandardScaler) achieved a test **MAE of 1.650 °C**, representing a major improvement over the persistence baseline (**3.144 °C**). This confirms that the recurrent architecture captures meaningful temporal dependencies in the meteorological data.

The **Best Evolutionary GRU** improved the baseline further, achieving **1.598 °C MAE** and **2.157 °C RMSE** on the held-out test set. This corresponds to a **3.2% reduction in MAE** relative to the baseline, while also using fewer trainable parameters.

The **Pruned Evolutionary GRU**, obtained by removing five low-importance features identified through XAI, delivered the best main final evaluation result: **1.575 °C MAE** and **2.134 °C RMSE**. This corresponds to a **4.5% improvement in MAE** over the baseline, while also reducing model size and input dimensionality.

### Impact of Evolutionary Optimization

The evolutionary search proved capable of improving the forecasting pipeline beyond the manually defined baseline. Importantly, the optimization did not focus only on GRU hyperparameters, but on the forecasting pipeline more broadly, including preprocessing, training settings, and windowing strategy.

The main discoveries of the search were:

- **RobustScaler** instead of StandardScaler, suggesting that the dataset benefits from a more outlier-resistant normalization strategy
- **MAE loss** instead of Huber loss, leading to strong optimization performance and direct alignment with the evaluation objective
- a **compact 2-layer GRU architecture** (`64 → 64`) rather than wider alternatives
- **144-hour lookback**, showing that the temporal context length should not be treated as fixed
- little or no benefit from additional dropout or Gaussian noise in the best-performing configurations

These results support the idea that evolutionary search can uncover leaner and better-generalizing forecasting pipelines than manual tuning alone.

### Robustness Validation

To mitigate the risk of validation-set overfitting during evolutionary selection, the selected optimized models were re-evaluated across multiple random seeds. This robustness analysis showed that the improvements were not dependent on a single favourable initialization.

| Model | MAE mean ± std (°C) | RMSE mean ± std (°C) |
|---|---:|---:|
| Best Evolutionary GRU | 1.593 ± 0.016 | 2.151 ± 0.023 |
| Pruned Evolutionary GRU | 1.579 ± 0.009 | 2.134 ± 0.010 |
| **Pruned Evolutionary GRU (best observed seed)** | **1.569** | **2.123** |

The multi-seed analysis confirms that the **Pruned Evolutionary GRU** is the strongest and most stable model in the project. Across seeds **7, 21, and 42**, it achieved an average test performance of **1.579 ± 0.009 °C MAE** and **2.134 ± 0.010 °C RMSE**, while the **best seed-specific result** reached **1.569 °C MAE** and **2.123 °C RMSE**. This was the best forecasting result observed in the project, and for this reason the **Pruned Evolutionary GRU** is selected as the final model.

### Explainability Insights

The explainability analysis revealed a coherent and physically plausible structure.

At the global level, **past temperature** was by far the most important predictor, followed by cyclical temporal variables such as **`doy_cos`**, **`hour_cos`**, and **`hour_sin`**. Among the meteorological variables, **maximum wind speed**, **humidity**, **pressure**, and selected wind-derived components also contributed meaningfully.

At the local level, gradient-based saliency confirmed that the model relied primarily on the **most recent segment of the input history**, with the strongest attribution concentrated on **`T (degC)`** and, in the heatmap, also visibly on **`p (mbar)`**. These findings are consistent with the short-term autoregressive nature of the forecasting task.

The five removed features — `doy_sin`, `gust_ratio`, `wd (deg)`, `wd_sin`, and `wy` — showed minimal contribution in the global analysis, supporting the pruning decision.

### Accuracy–Efficiency Trade-offs

The optimized models achieved a favourable trade-off between predictive accuracy and computational cost.

Compared with the baseline, the evolutionary models:
- used fewer trainable parameters
- achieved lower test error
- maintained competitive inference latency
- and, after pruning, further reduced input dimensionality without sacrificing accuracy

This is an important result: the best-performing models were not larger or more complex than the hand-tuned baseline. On the contrary, evolutionary optimization and XAI-guided pruning led to a forecasting pipeline that is simultaneously **more accurate**, **more compact**, and **more efficient**.

### Strengths and Limitations

**Strengths**
- End-to-end forecasting pipeline with temporal split, leakage prevention, and untouched test set
- Evolutionary optimization applied to preprocessing, architecture, training, and windowing
- Explicit optimization of lookback length, satisfying the requirement that windowing should not remain fixed
- Multi-seed robustness analysis, providing average performance and variability instead of relying on a single run
- Global and local XAI analysis, followed by XAI-guided feature pruning
- Final model that is more accurate, simpler, and more efficient than the baseline

**Limitations**
- The evolutionary budget, although effective, still explores only a small portion of the overall search space
- The forecast horizon (`H = 24`) was kept fixed; only the lookback length was optimized
- Only the GRU family was explored; transformer-style or attention-based forecasting models were not included
- The TimeGAN component achieved strong reconstruction but insufficient synthetic realism, and therefore was not incorporated into the final forecasting pipeline

## 14. Future Work

Although the final forecasting pipeline achieved strong results, several directions remain open for future improvement.

First, the evolutionary search could be extended with a larger computational budget, allowing a broader exploration of the search space and potentially identifying even stronger configurations. In particular, future work could include a wider range of lookback values, alternative forecast horizons, and richer multi-objective formulations balancing accuracy, complexity, and efficiency.

Second, the modelling space could be expanded beyond GRU-based architectures. Transformer-based models, temporal convolutional networks, hybrid recurrent–attention models, or lightweight sequence models could provide useful benchmarks and reveal whether the observed gains generalize beyond a single architecture family.

Third, the robustness analysis could be extended further by increasing the number of random seeds and by evaluating the best configurations under rolling-origin or multi-period backtesting schemes. This would provide a stronger estimate of model stability under different temporal regimes.

Fourth, the TimeGAN component deserves further investigation. In this project, the synthetic sequences showed good reconstruction quality but insufficient realism for safe integration into the final forecasting pipeline. Future work could explore improved generative architectures, longer adversarial training, alternative conditioning strategies, or diffusion-based time-series generators to assess whether synthetic augmentation can become beneficial.

Fifth, the XAI-guided pruning results suggest that explainability can be used not only for interpretation but also for model refinement. This idea could be extended into an iterative explainability–selection loop, where feature importance is repeatedly re-estimated and the input space is progressively simplified while monitoring predictive performance and robustness.

Finally, future work could place stronger emphasis on deployment-oriented evaluation, including more precise GPU profiling, energy consumption, memory peaks, and latency under real-time forecasting constraints. This would help determine whether the final model is not only accurate and interpretable, but also suitable for practical operational use.

## 15. Conclusion

This project developed a comprehensive end-to-end forecasting pipeline for **24-hour air temperature prediction** on the **Jena Climate dataset**, integrating **evolutionary optimization**, **explainability**, and **efficiency analysis** within a unified experimental framework. An optional **TimeGAN** component was also explored as a proof-of-concept for synthetic data generation, although it was not retained in the final forecasting pipeline due to insufficient synthetic realism.

### Key Findings

The official **GRU baseline**, based on the best-performing configuration from the previous mini-project, achieved a test **MAE of 1.650 °C** and **RMSE of 2.193 °C**, representing a substantial improvement over the naive persistence forecast (**3.144 °C MAE**). This established a strong and competitive reference model.

The **evolutionary algorithm**, applied to the forecasting pipeline as a whole rather than to isolated hyperparameters, discovered a better-performing configuration while also reducing model complexity. The search showed that:
- **RobustScaler** outperformed StandardScaler for this task
- **MAE loss** outperformed the baseline Huber setup
- a **compact 2-layer GRU architecture** (`64 → 64`) generalized better than the larger manually defined baseline
- a **144-hour lookback** was preferred over the default 120-hour setting, confirming that the windowing strategy should not be treated as fixed

The **Best Evolutionary GRU** achieved a test **MAE of 1.598 °C** and **RMSE of 2.157 °C**, improving on the baseline while using fewer trainable parameters.

### XAI-Driven Model Refinement

A distinctive contribution of this project is the use of explainability not only for interpretation, but also for **model refinement**. Global permutation importance and local saliency analysis consistently showed that recent temperature history dominated the forecasts, while a small subset of variables had negligible influence.

Based on these explainability results, five low-importance features (`doy_sin`, `gust_ratio`, `wd (deg)`, `wd_sin`, and `wy`) were removed and the optimized model was retrained on the reduced feature set. This produced the best main final evaluation result: the **Pruned Evolutionary GRU** achieved **1.575 °C MAE** and **2.134 °C RMSE**, corresponding to a **4.5% MAE improvement over the baseline**.

Multi-seed robustness analysis further strengthened this conclusion. Across seeds **7, 21, and 42**, the pruned model achieved **1.579 ± 0.009 °C MAE**, compared with **1.593 ± 0.016 °C** for the full-feature evolutionary model. The **best observed seed-specific result** reached approximately **1.569 °C MAE**, which was the strongest forecasting result obtained in the project. This closed-loop methodology — **optimize → explain → prune → validate** — proved to be one of the most important outcomes of the work.

### Final Assessment

Overall, the project shows that a forecasting system can be improved substantially through the combination of:
- a strong baseline model
- end-to-end evolutionary optimization
- global and local explainability
- XAI-guided feature selection
- robustness and efficiency analysis

The final result is a forecasting pipeline that is **more accurate, more compact, more interpretable, and more efficient** than the manually defined baseline. The **Pruned Evolutionary GRU** is therefore selected as the final model of the project, since it achieved the strongest main evaluation result and also produced the best observed forecasting run (**1.569 °C MAE**) during the multi-seed robustness analysis.